# Projeto PjBL — Acidentes em rodovias federais (2025)

**Grupo:** Giovanna Rita Santos Daniel; João Lucas do Prado Ribeiro; Leonardo Stall; Mirella Haisi Szvarça  
**Turma:** 6A

**Problema investigado:** compreender a distribuição dos acidentes em rodovias federais brasileiras, suas características temporais e de gravidade e a participação de condutores menores de idade. A base combina informações de acidentes, pessoas, veículos, causas e tipos de ocorrência. Por isso, antes de contar eventos ou envolvidos, é necessário identificar a unidade representada por cada registro.

**Método:** leitura e exploração da fonte, diagnóstico dos problemas de qualidade, limpeza com regras explícitas, transformação das variáveis necessárias e verificação da integridade dos dados resultantes. Todo o processamento é implementado em PySpark.


# Organização e decisões de processamento

Os dados são mantidos em três estágios para que os resultados possam ser conferidos sem sobrescrever a fonte. `df_original` preserva os registros lidos; `df_tratado` reúne as correções de ausência, plausibilidade e tipagem; `df_transformado` acrescenta variáveis úteis à análise. Conjuntos auxiliares representam acidentes, combinações acidente–tipo e pessoas identificadas, respeitando suas diferentes granularidades.

Essa separação evita que uma correção destrua a evidência original e permite comparar a quantidade de registros, os valores alterados e os tipos dos campos entre os estágios. As cinco perguntas de pesquisa direcionam as variáveis preparadas, sem substituir a verificação da qualidade por conclusões antecipadas.

**Cuidado de interpretação:** cada linha não representa necessariamente um acidente ou uma pessoa diferente. `mortos`, `ilesos`, `feridos_leves` e `feridos_graves` são indicadores 0/1 associados à pessoa na linha, não contagens totais do acidente.


## 1. Contexto e objetivo

Os acidentes em rodovias federais envolvem fatores geográficos, temporais, ambientais, veiculares e humanos. O problema investigado é compreender como essas características se relacionam com a ocorrência e a gravidade dos acidentes registrados em 2025. Antes de fazer comparações, é necessário reconhecer que a base é **agrupada por pessoa e por múltiplas causas/tipos**: o mesmo acidente pode aparecer em várias linhas. Contar linhas como se fossem acidentes produziria resultados incorretos.

**Perguntas de pesquisa que orientam a preparação:**

1. Como a quantidade de acidentes varia entre UFs e rodovias federais no período selecionado?
2. Como as ocorrências se distribuem entre meses, dias da semana e horários?
3. Como a proporção de acidentes com vítimas fatais varia entre os tipos de acidente?
4. Como a gravidade dos acidentes registrados varia entre condições meteorológicas e fases do dia?
5. Como a participação de condutores menores de idade nos acidentes varia entre tipos de veículo e UFs?

**Objetivo desta entrega:** carregar o arquivo, compreender sua estrutura e qualidade, justificar os tratamentos aplicados e criar transformações úteis às perguntas, preservando a granularidade e distinguindo informações ausentes de atributos não aplicáveis. As respostas quantitativas às perguntas serão desenvolvidas em análises posteriores.

**Unidade de observação.** `id` identifica um acidente; `pesid` representa uma pessoa identificada, mas nem toda linha possui esse identificador; `ordem_tipo_acidente` diferencia registros de tipo no mesmo acidente. A ausência de `pesid` não justifica eliminar automaticamente informações sobre o acidente. Por isso, cada cálculo e possível deduplicação exige uma chave coerente com a unidade estudada.

## 2. Fonte e ingestão dos dados

Esta seção identifica a procedência, o formato e o significado do arquivo e demonstra uma leitura reprodutível em PySpark. A decisão de **ler inicialmente tudo como texto** é intencional: impede que a inferência automática altere identificadores, converta marcadores de ausência sem controle ou interprete de modo errado decimais escritos com vírgula. Os tipos apropriados são atribuídos após a investigação e a limpeza, na seção 4.

**Fonte e caracterização dos dados**

- **Produtor:** Polícia Rodoviária Federal (PRF), base de dados de acidentes (BAT).
- **Catálogo oficial:** https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dados-abertos-da-prf
- **Recurso:** Documento CSV de Acidentes 2025 — *Agrupados por pessoa (Todas as causas e tipos de acidentes)*.
- **Dicionário oficial:** https://www.gov.br/prf/pt-br/acesso-a-informacao/dados-abertos/dicionario-acidentes
- **Arquivo de trabalho:** `acidentes2025_todas_causas_tipos.csv`.
- **Referência de acesso:** portal oficial da PRF, disponível no endereço acima. A data exata do download do grupo não está comprovada pelo arquivo e, portanto, não é atribuída.
- **Formato:** CSV delimitado por `;`, codificação `ISO-8859-1`, números decimais com vírgula, cabeçalho na primeira linha.
- **Dimensão verificada no arquivo anexado:** 584.010 linhas e 37 colunas, a serem reconfirmadas em PySpark na seção 3.1.
- **Atributos principais:** `id` (acidente), `pesid` (pessoa), `id_veiculo`, `data_inversa`, `horario`, `uf`, `br`, `km`, `tipo_acidente`, `classificacao_acidente`, `condicao_metereologica`, `fase_dia`, `tipo_envolvido`, `idade` e `tipo_veiculo`.

**Justificativa da ingestão:** a leitura inicial como `StringType` evita inferências prematuras e conserva marcadores de ausência, zeros e formatos necessários à auditoria. A conversão definitiva acontece apenas após a normalização e a validação na seção 4.

### 2.1 — Leitura controlada do CSV

O código cria a sessão Spark e lê o CSV com **delimitador `;` e codificação ISO-8859-1**, compatíveis com a fonte. O cabeçalho é conferido antes de qualquer análise para evitar conclusões sobre um arquivo incompleto ou trocado. `inferSchema=False` preserva os valores como recebidos; `mode="FAILFAST"` sinaliza registros malformados em vez de descartá-los silenciosamente.

**Reprodutibilidade:** `CAMINHO_DADOS` pode ser configurado por variável de ambiente; `/content/acidentes2025_todas_causas_tipos.csv` é somente o valor padrão no Colab. Nesse ambiente, instale PySpark caso necessário (por exemplo, `%pip install pyspark`) e disponibilize o CSV no caminho configurado. Nenhuma instalação é acionada automaticamente no notebook.

In [1]:
from pathlib import Path
from pyspark.sql import SparkSession, functions as F

def percentual(parte, total):
    return round((float(parte) / total * 100), 2) if total else 0.0

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("PjBL_RA1_Acidentes_2025")
    .getOrCreate()
)

# No Colab, carregue o CSV em /content. Fora dele, configure a variável de ambiente.
import os
CAMINHO_DADOS = os.environ.get(
    "CAMINHO_DADOS",
    "/content/acidentes2025_todas_causas_tipos.csv"
)

if not Path(CAMINHO_DADOS).is_file():
    raise FileNotFoundError(f"Arquivo não encontrado: {CAMINHO_DADOS}. Faça upload no Colab ou defina CAMINHO_DADOS no ambiente.")

df_original = spark.read.options(
    header=True,
    sep=";",
    encoding="ISO-8859-1",
    inferSchema=False,
    quote='"',
    escape='"',
    multiLine=True,
    mode="FAILFAST"
).csv(CAMINHO_DADOS)

colunas_obrigatorias = {"id", "pesid", "data_inversa", "horario", "uf", "br", "km", "tipo_acidente", "classificacao_acidente", "tipo_envolvido", "idade", "id_veiculo", "tipo_veiculo", "marca", "ano_fabricacao_veiculo", "condicao_metereologica", "fase_dia"}
if (not df_original.columns or any(not coluna.strip() for coluna in df_original.columns)
        or not colunas_obrigatorias.issubset(set(df_original.columns))):
    raise ValueError("Cabeçalho incompatível com a base esperada; confira delimitador, arquivo e upload.")

print(f"Fonte de dados: {CAMINHO_DADOS}")
print("Sessão Spark iniciada e cabeçalho verificado.")


Fonte de dados: /content/acidentes2025_todas_causas_tipos.csv
Sessão Spark iniciada e cabeçalho verificado.


### 2.2 — Dicionário de dados e relação com as perguntas

A tabela identifica **cada uma das 37 colunas originais**, apresenta seu **tipo de armazenamento pretendido após a limpeza**, explica seu significado e marca a relação direta com as perguntas P1–P5. Essa leitura permite verificar se o arquivo contém os atributos necessários e reconhecer que o **tipo físico** (por exemplo, `string`) não deve ser confundido com o significado do campo. O mapeamento não adiciona nem modifica atributos.

In [2]:
# Mapeamento dos atributos existentes: apoio conceitual, sem criar colunas no CSV.
# Perguntas P1...P5 seguem a numeração da seção 1.
mapa = {
    "P1": {"id", "uf", "br", "data_inversa"},
    "P2": {"id", "data_inversa", "dia_semana", "horario"},
    "P3": {"id", "tipo_acidente", "ordem_tipo_acidente", "classificacao_acidente", "mortos", "pesid"},
    "P4": {"id", "classificacao_acidente", "condicao_metereologica", "fase_dia"},
    "P5": {"id", "pesid", "tipo_envolvido", "idade", "tipo_veiculo", "id_veiculo", "uf"},
}
significados = {
    "id":"Identificador do acidente", "pesid":"Identificador da pessoa", "data_inversa":"Data do acidente", "dia_semana":"Dia da semana", "horario":"Hora informada", "uf":"Unidade federativa", "br":"Número da rodovia federal", "km":"Marco quilométrico", "municipio":"Município", "causa_principal":"Indicador da causa principal", "causa_acidente":"Causa registrada", "ordem_tipo_acidente":"Ordem de registro do tipo de acidente", "tipo_acidente":"Tipo do acidente", "classificacao_acidente":"Classificação de gravidade do acidente", "fase_dia":"Condição de iluminação do período", "sentido_via":"Sentido de circulação", "condicao_metereologica":"Condição meteorológica", "tipo_pista":"Tipo de pista", "tracado_via":"Características do traçado", "uso_solo":"Uso do solo", "id_veiculo":"Identificador do veículo", "tipo_veiculo":"Categoria do veículo", "marca":"Marca do veículo", "ano_fabricacao_veiculo":"Ano de fabricação", "tipo_envolvido":"Papel da pessoa envolvida", "estado_fisico":"Estado físico informado", "idade":"Idade em anos", "sexo":"Sexo informado", "ilesos":"Indicador individual de pessoa ilesa", "feridos_leves":"Indicador individual de ferimento leve", "feridos_graves":"Indicador individual de ferimento grave", "mortos":"Indicador individual de morte", "latitude":"Coordenada geográfica", "longitude":"Coordenada geográfica", "regional":"Unidade regional PRF", "delegacia":"Delegacia PRF", "uop":"Unidade operacional PRF"
}
inteiros_mapa = {"br", "ordem_tipo_acidente", "ano_fabricacao_veiculo", "idade", "ilesos", "feridos_leves", "feridos_graves", "mortos"}
decimais_mapa = {"km", "latitude", "longitude"}
def tipo_previsto(c):
    return "date" if c == "data_inversa" else "integer" if c in inteiros_mapa else "double" if c in decimais_mapa else "string"
linhas_mapa = [
    (c, tipo_previsto(c), significados.get(c, "Atributo original da PRF"), *[("Sim" if c in mapa[p] else "—") for p in ["P1", "P2", "P3", "P4", "P5"]])
    for c in df_original.columns
]
df_mapa = spark.createDataFrame(linhas_mapa, "atributo string, tipo_previsto string, significado string, P1 string, P2 string, P3 string, P4 string, P5 string")
print("Dicionário dos 37 atributos e correspondência com as cinco perguntas:")
df_mapa.show(len(linhas_mapa), truncate=False)
print("Nota: o mapa indica relevância direta, não dispensa outras colunas para verificar granularidade.")

Dicionário dos 37 atributos e correspondência com as cinco perguntas:
+----------------------+-------------+---------------------------------------+---+---+---+---+---+
|atributo              |tipo_previsto|significado                            |P1 |P2 |P3 |P4 |P5 |
+----------------------+-------------+---------------------------------------+---+---+---+---+---+
|id                    |string       |Identificador do acidente              |Sim|Sim|Sim|Sim|Sim|
|pesid                 |string       |Identificador da pessoa                |—  |—  |Sim|—  |Sim|
|data_inversa          |date         |Data do acidente                       |Sim|Sim|—  |—  |—  |
|dia_semana            |string       |Dia da semana                          |—  |Sim|—  |—  |—  |
|horario               |string       |Hora informada                         |—  |Sim|—  |—  |—  |
|uf                    |string       |Unidade federativa                     |Sim|—  |—  |—  |Sim|
|br                    |integer      |N

## 3. Exploração inicial dos dados

A exploração utiliza **`df_original`, sem corrigi-lo**, para documentar os problemas efetivamente encontrados e fundamentar cada decisão da seção 4. São examinados: estrutura, ausências físicas e textuais, categorias, valores numéricos suspeitos, regras de plausibilidade e repetições dos identificadores. O objetivo não é eliminar registros com aparente anomalia, mas distinguir erros verificáveis de situações legítimas da extração.

As contagens de `id`, `pesid` e dos indicadores por pessoa requerem interpretação cuidadosa: linhas repetidas por chaves parciais podem representar fatos diferentes e não devem ser removidas automaticamente.

#### 3.1 — Estrutura e duplicações exatas

**O que:** identificar as dimensões da amostra, os nomes e tipos inicialmente lidos e possíveis linhas integralmente idênticas. **Por quê:** essa é a referência para conferir, ao final, se o tratamento preservou os registros. Linhas com o mesmo `id` ou `pesid` **não são automaticamente duplicatas**: o arquivo foi disponibilizado com detalhamento por pessoa, causa e tipo. Por isso, investigamos repetições de chave separadamente na seção 3.6 e só consideraríamos excluir cópias idênticas mediante evidência de erro, que não apareceu na exploração.

In [3]:

# 3.1 — Dimensões, estrutura e duplicidades exatas
# Reutilizamos o total de registros para evitar contagens repetidas desnecessárias.
total_registros = df_original.count()
total_colunas = len(df_original.columns)

# Uma única ação para avaliar duplicatas exatas.
total_duplicados = total_registros - df_original.dropDuplicates().count()

print(f"Total de registros: {total_registros:,}")
print(f"Total de colunas: {total_colunas}")
print(f"Registros completamente duplicados: {total_duplicados:,}")
print("\nEsquema:")
df_original.printSchema()

print("\nAmostra:")
df_original.show(20, truncate=False)


Total de registros: 584,010
Total de colunas: 37
Registros completamente duplicados: 0

Esquema:
root
 |-- id: string (nullable = true)
 |-- pesid: string (nullable = true)
 |-- data_inversa: string (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: string (nullable = true)
 |-- km: string (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_principal: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- ordem_tipo_acidente: string (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nullable = true)
 |-- id_veiculo: string (nullable = true)
 |-- tipo_ve

### 3.2 — Nulos físicos e marcadores textuais

**Por que avaliar os dois?** O CSV usa mais de uma representação de desconhecimento: `NULL` físico, `NA`, vazios, `Ignorado`, `Não Informado` e outras grafias. Contar apenas `isNull()` antes da normalização **subestima** o problema. O diagnóstico mantém separadas as contagens de nulos físicos e marcadores textuais para justificar a padronização da seção 4.2. Essa etapa é apenas descritiva: a função `campo_ausente()` identifica as situações, mas não altera `df_original`.

In [4]:
# 3.2 — Valores ausentes: nulos reais e marcadores textuais
# Mantemos a base original intacta. A função abaixo permite ignorar ausências
# tanto nas contagens de IDs quanto nas demais verificações.
marcadores_ausencia = [
    "", "NA", "N/A", "NULL", "NONE", "NAN", "-", "--",
    "IGNORADO", "NÃO INFORMADO", "NAO INFORMADO",
    "NA/NA", "NÃO INFORMADO/NÃO INFORMADO"
]
marcadores_normalizados = [m.upper() for m in marcadores_ausencia]

def campo_ausente(nome):
    texto = F.upper(F.trim(F.col(nome).cast("string")))
    return F.col(nome).isNull() | texto.isin(marcadores_normalizados)

# Nulos reais e marcadores textuais são reportados separadamente para não
# confundir ausência física com valores codificados como texto.
contagens_nulas = df_original.agg(*[
    F.count(F.when(F.col(c).isNull(), 1)).alias(f"n{i}")
    for i, c in enumerate(df_original.columns)
]).first()
linhas_nulos = [
    (c, int(contagens_nulas[f"n{i}" ] or 0),
     percentual(contagens_nulas[f"n{i}" ] or 0, total_registros))
    for i, c in enumerate(df_original.columns)
]
df_nulos = spark.createDataFrame(
    linhas_nulos,
    "coluna string, quantidade_nula long, percentual_nulo double"
).orderBy(F.desc("percentual_nulo"))
print("Nulos reais por coluna:")
df_nulos.show(len(df_original.columns), truncate=False)

expressoes_marcadores = []
for i, c in enumerate(df_original.columns):
    normalizado = F.upper(F.trim(F.col(c).cast("string")))
    expressoes_marcadores.append(
        F.sum(F.when(normalizado.isin(marcadores_normalizados), 1).otherwise(0)).alias(f"m{i}")
    )
contagens_marcadores = df_original.agg(*expressoes_marcadores).first()
linhas_marcadores = [
    (c, int(contagens_marcadores[f"m{i}"] or 0),
     percentual(int(contagens_marcadores[f"m{i}"] or 0), total_registros))
    for i, c in enumerate(df_original.columns)
    if int(contagens_marcadores[f"m{i}"] or 0) > 0
]
df_marcadores = spark.createDataFrame(
    linhas_marcadores,
    "coluna string, ocorrencias_marcadores long, percentual_registros double"
).orderBy(F.desc("ocorrencias_marcadores"))
print("Ocorrências de possíveis marcadores textuais de ausência (inclui strings vazias):")
df_marcadores.show(max(len(linhas_marcadores), 1), truncate=False)


Nulos reais por coluna:
+----------------------+---------------+---------------+
|coluna                |quantidade_nula|percentual_nulo|
+----------------------+---------------+---------------+
|tracado_via           |0              |0.0            |
|id                    |0              |0.0            |
|uso_solo              |0              |0.0            |
|pesid                 |0              |0.0            |
|id_veiculo            |0              |0.0            |
|data_inversa          |0              |0.0            |
|tipo_veiculo          |0              |0.0            |
|dia_semana            |0              |0.0            |
|marca                 |0              |0.0            |
|horario               |0              |0.0            |
|ano_fabricacao_veiculo|0              |0.0            |
|uf                    |0              |0.0            |
|tipo_envolvido        |0              |0.0            |
|br                    |0              |0.0            |
|estado

#### 3.3 — Categorias existentes

**O que:** mostrar as categorias mais frequentes dos atributos relevantes. **Por quê:** verificar a estrutura dos valores antes de modificá-los e identificar grafias e rótulos de desconhecimento que não representam categorias observadas. As frequências são **de linhas da extração**, não de acidentes distintos; portanto, ainda não respondem às perguntas analíticas.

In [5]:
# 3.3 — Distribuição de variáveis categóricas
# Para facilitar a leitura, exibimos as 10 categorias mais frequentes por variável.
# A distribuição completa continua disponível na variável distribuicao da iteração.
colunas_categoricas = [
    "uf", "dia_semana", "classificacao_acidente", "tipo_acidente",
    "causa_principal", "causa_acidente", "fase_dia",
    "condicao_metereologica", "tipo_pista", "sentido_via",
    "uso_solo", "tipo_veiculo", "tipo_envolvido",
    "estado_fisico", "sexo"
]
colunas_categoricas = [c for c in colunas_categoricas if c in df_original.columns]

for coluna in colunas_categoricas:
    print(f"\nDistribuição de {coluna}:")
    distribuicao = (
        df_original.groupBy(coluna)
        .count()
        .withColumn(
            "percentual_registros",
            F.round(F.col("count") / F.lit(total_registros) * 100, 2)
        )
        .orderBy(F.desc("count"))
    )
    quantidade_categorias = distribuicao.count()
    print(f"Quantidade de categorias distintas (incluindo nulos): {quantidade_categorias}")
    print("10 categorias mais frequentes:")
    distribuicao.show(10, truncate=False)

# Verificar grafias potencialmente inconsistentes em campos de texto selecionados.
# A normalização é apenas diagnóstica; os valores originais não são alterados.
# Para evitar saídas excessivas, mostramos no máximo 20 grupos de variações.
for coluna in ["marca", "municipio", "causa_acidente", "tipo_acidente"]:
    if coluna not in df_original.columns:
        continue
    valores = (
        df_original.select(F.trim(F.col(coluna)).alias("valor"))
        .filter(F.col("valor").isNotNull())
        .distinct()
    )
    chave = F.lower(F.regexp_replace(F.col("valor"), r"\s+", " "))
    variacoes = (
        valores.withColumn("chave_normalizada", chave)
        .groupBy("chave_normalizada")
        .agg(F.collect_set("valor").alias("grafias"))
        .filter(F.size("grafias") > 1)
    )
    quantidade_variacoes = variacoes.count()
    if quantidade_variacoes > 0:
        print(f"\nPossíveis variações de escrita em {coluna}: {quantidade_variacoes} grupos")
        variacoes.show(20, truncate=False)



Distribuição de uf:
Quantidade de categorias distintas (incluindo nulos): 27
10 categorias mais frequentes:
+---+-----+--------------------+
|uf |count|percentual_registros|
+---+-----+--------------------+
|MG |82397|14.11               |
|PR |70759|12.12               |
|SC |51403|8.8                 |
|RS |35952|6.16                |
|BA |33498|5.74                |
|SP |32280|5.53                |
|RJ |31474|5.39                |
|GO |30070|5.15                |
|MT |26891|4.6                 |
|PE |25046|4.29                |
+---+-----+--------------------+
only showing top 10 rows

Distribuição de dia_semana:
Quantidade de categorias distintas (incluindo nulos): 7
10 categorias mais frequentes:
+-------------+-----+--------------------+
|dia_semana   |count|percentual_registros|
+-------------+-----+--------------------+
|sábado       |93778|16.06               |
|domingo      |93672|16.04               |
|sexta-feira  |93659|16.04               |
|segunda-feira|80118|13.72    

### 3.4 — Leitura numérica para diagnóstico

Idade, quilometragem, coordenadas, ano de fabricação e indicadores de vítimas chegam inicialmente como strings. Esta célula produz conversões **auxiliares de diagnóstico** para localizar valores malformados e resumir os valores reconhecíveis. Ainda não transforma a base entregue. Os quatro campos de vítimas devem ser interpretados como indicadores **0/1 por pessoa**, e não como quantidades agregadas de mortes ou feridos no acidente.

In [6]:

# 3.4 — Conversão diagnóstica e estatísticas numéricas
# A base original permanece inalterada. As colunas *_num são criadas somente para análise.
colunas_previstas = [
    "idade", "ano_fabricacao_veiculo", "km", "br",
    "latitude", "longitude", "ilesos",
    "feridos_leves", "feridos_graves", "mortos"
]
colunas_numericas = [c for c in colunas_previstas if c in df_original.columns]

df_numerico = df_original
for coluna in colunas_numericas:
    numero = F.expr(
        f"try_cast(replace(trim(`{coluna}`), ',', '.') AS DOUBLE)"
    )
    valido = (
        numero.isNotNull()
        & ~F.isnan(numero)
        & (F.abs(numero) != float("inf"))
    )
    df_numerico = df_numerico.withColumn(
        coluna + "_num", F.when(valido, numero)
    )

# Resumo de conversões em uma única agregação.
expressoes_conversao = []
for i, coluna in enumerate(colunas_numericas):
    expressoes_conversao.extend([
        F.count(F.col(coluna)).alias(f"orig_{i}"),
        F.count(F.col(coluna + "_num")).alias(f"conv_{i}")
    ])
contagens_conv = df_numerico.agg(*expressoes_conversao).first() if expressoes_conversao else None

linhas_conv = []
for i, coluna in enumerate(colunas_numericas):
    originais = int(contagens_conv[f"orig_{i}"] or 0)
    convertidos = int(contagens_conv[f"conv_{i}"] or 0)
    falhas = originais - convertidos
    linhas_conv.append((coluna, originais, convertidos, falhas,
                        percentual(falhas, originais)))
df_conversao = spark.createDataFrame(
    linhas_conv,
    "coluna string, valores_nao_nulos long, valores_convertidos long, "
    "falhas_conversao long, percentual_falhas double"
)
print("Resumo de conversão numérica:")
df_conversao.show(max(len(linhas_conv), 1), truncate=False)

colunas_estatisticas = [
    c for c in ["idade", "ano_fabricacao_veiculo", "km", "br",
                "ilesos", "feridos_leves", "feridos_graves", "mortos"]
    if c in colunas_numericas
]
if colunas_estatisticas:
    df_numerico.select(*[
        F.col(c + "_num").alias(c) for c in colunas_estatisticas
    ]).summary("count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show(8, truncate=False)

# Complemento solicitado: verificar se os campos que devem ser inteiros têm frações.
# A conversão e as estatísticas acima continuam iguais às da versão anterior.
colunas_inteiras_esperadas = [
    c for c in ["idade", "ano_fabricacao_veiculo", "br", "ilesos",
                "feridos_leves", "feridos_graves", "mortos"]
    if c in colunas_numericas
]

linhas_inteiros = []
for coluna in colunas_inteiras_esperadas:
    valor_num = F.col(coluna + "_num")
    fracionarios = (
        df_numerico
        .filter(valor_num.isNotNull() & (valor_num != F.floor(valor_num)))
        .count()
    )
    linhas_inteiros.append((coluna, fracionarios))

df_validacao_inteiros = spark.createDataFrame(
    linhas_inteiros,
    "coluna string, valores_com_parte_decimal long"
)
print("Verificação adicional: valores fracionários em colunas que deveriam ser inteiras:")
df_validacao_inteiros.show(max(len(linhas_inteiros), 1), truncate=False)

# Mostra exemplos de conteúdo preenchido que não pôde ser convertido em número.
# A contagem total de falhas já está apresentada no resumo df_conversao acima.
for coluna in colunas_numericas:
    if df_conversao.filter(
        (F.col("coluna") == coluna) & (F.col("falhas_conversao") > 0)
    ).count() == 0:
        continue
    exemplos = (
        df_original
        .filter(F.col(coluna).isNotNull())
        .withColumn(
            "_numero_teste",
            F.expr(f"try_cast(replace(trim(`{coluna}`), ',', '.') AS DOUBLE)")
        )
        .filter(F.col("_numero_teste").isNull())
        .groupBy(F.trim(F.col(coluna)).alias("valor_original"))
        .count()
        .orderBy(F.desc("count"))
    )
    print(f"Exemplos de valores que falharam na conversão de {coluna}:")
    exemplos.show(10, truncate=False)



Resumo de conversão numérica:
+----------------------+-----------------+-------------------+----------------+-----------------+
|coluna                |valores_nao_nulos|valores_convertidos|falhas_conversao|percentual_falhas|
+----------------------+-----------------+-------------------+----------------+-----------------+
|idade                 |584010           |478397             |105613          |18.08            |
|ano_fabricacao_veiculo|584010           |565393             |18617           |3.19             |
|km                    |584010           |582566             |1444            |0.25             |
|br                    |584010           |582566             |1444            |0.25             |
|latitude              |584010           |584010             |0               |0.0              |
|longitude             |584010           |584010             |0               |0.0              |
|ilesos                |584010           |527209             |56801           |9.73     

#### 3.5 — Valores suspeitos e verificações de domínio

**O que:** conferir idade, ano do veículo, limites numéricos, datas, horários e coordenadas. **Por quê:** valor estatisticamente incomum e valor impossível são coisas diferentes. KM alto indica uma posição na rodovia, não um acidente anormal; idade extremamente alta pode representar erro de digitação. Nesta fase só identificamos ocorrências suspeitas, sem removê-las. O tratamento justificado virá na seção 4.

In [7]:
# 3.5 — Valores suspeitos, validação de datas, KM, horários e coordenadas
criterios = []

if "idade" in colunas_numericas:
    idade = F.col("idade_num")
    criterios += [
        ("Idade negativa", idade < 0),
        ("Idade igual a zero — verificar contexto", idade == 0),
        ("Idade acima de 110 — verificar", idade > 110)
    ]

if "ano_fabricacao_veiculo" in colunas_numericas:
    ano = F.col("ano_fabricacao_veiculo_num")
    criterios += [
        ("Ano de fabricação igual a zero", ano == 0),
        ("Ano de fabricação negativo", ano < 0)
    ]
    if "data_inversa" in df_original.columns:
        data_acidente = F.expr("try_cast(trim(`data_inversa`) AS DATE)")
        criterios.append((
            "Ano de fabricação posterior ao acidente — verificar",
            data_acidente.isNotNull() & (ano > F.year(data_acidente))
        ))

if "km" in colunas_numericas:
    km = F.col("km_num")
    criterios += [
        ("Quilometragem negativa", km < 0),
        ("Quilometragem igual a zero — verificar", km == 0),
        ("Quilometragem acima de 1000 km — revisar, sem exclusão automática", km > 1000)
    ]

for coluna in ["ilesos", "feridos_leves", "feridos_graves", "mortos"]:
    if coluna in colunas_numericas:
        criterios.append((f"Valor negativo em {coluna}", F.col(coluna + "_num") < 0))

if criterios:
    contagens = df_numerico.agg(*[
        F.count(F.when(condicao, 1)).alias(f"v{i}")
        for i, (_, condicao) in enumerate(criterios)
    ]).first()
    df_dominios = spark.createDataFrame(
        [(nome, int(contagens[f"v{i}"] or 0)) for i, (nome, _) in enumerate(criterios)],
        "verificacao string, quantidade long"
    )
    print("Verificações de domínio (sinalizadores para investigação; não excluem registros):")
    df_dominios.show(len(criterios), truncate=False)

# Datas: separa ausência de valor de texto preenchido que não pôde ser convertido.
if "data_inversa" in df_original.columns:
    data_texto = F.trim(F.col("data_inversa"))
    data = F.expr("try_cast(trim(`data_inversa`) AS DATE)")
    df_original.agg(
        F.min(data).alias("primeira_data_valida"),
        F.max(data).alias("ultima_data_valida"),
        F.count(F.when(campo_ausente("data_inversa"), 1)).alias("datas_ausentes"),
        F.count(F.when(~campo_ausente("data_inversa") & data.isNull(), 1)).alias("datas_preenchidas_invalidas"),
        F.count(F.when(data.isNotNull() & (F.year(data) != 2025), 1)).alias("datas_fora_de_2025")
    ).show(1, truncate=False)

# Horários: aceita HH:mm ou HH:mm:ss, após remover espaços nas extremidades.
if "horario" in df_original.columns:
    horario_texto = F.trim(F.col("horario"))
    formato_hora = horario_texto.rlike(r"^(?:[01][0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9])?$")
    df_original.agg(
        F.count(F.when(campo_ausente("horario"), 1)).alias("horarios_ausentes"),
        F.count(F.when(~campo_ausente("horario") & ~formato_hora, 1)).alias("horarios_formato_invalido")
    ).show(1, truncate=False)

# Coordenadas: conversão numérica, ausências, limites mundiais, faixa aproximada do Brasil e zeros.
if {"latitude", "longitude"}.issubset(df_original.columns):
    coords = (
        df_original
        .withColumn("latitude_num", F.expr("try_cast(replace(trim(`latitude`), ',', '.') AS DOUBLE)"))
        .withColumn("longitude_num", F.expr("try_cast(replace(trim(`longitude`), ',', '.') AS DOUBLE)"))
    )
    lat, lon = F.col("latitude_num"), F.col("longitude_num")
    lat_ausente, lon_ausente = campo_ausente("latitude"), campo_ausente("longitude")
    par_valido = lat.isNotNull() & lon.isNotNull()
    limites_mundiais = lat.between(-90, 90) & lon.between(-180, 180)
    brasil_aprox = lat.between(-34, 6) & lon.between(-74, -28)
    coords.agg(
        F.count(F.when(lat_ausente | lon_ausente | lat.isNull() | lon.isNull(), 1)).alias("pares_ausentes_ou_invalidos"),
        F.count(F.when(par_valido & ~limites_mundiais, 1)).alias("fora_limites_mundiais"),
        F.count(F.when(par_valido & limites_mundiais & ~brasil_aprox, 1)).alias("fora_retangulo_aproximado_brasil"),
        F.count(F.when(par_valido & (lat == 0) & (lon == 0), 1)).alias("pares_iguais_a_zero")
    ).show(1, truncate=False)
    print("Observação: a faixa do Brasil é aproximada; coordenadas fora dela devem ser revisadas, não excluídas automaticamente.")


Verificações de domínio (sinalizadores para investigação; não excluem registros):
+-----------------------------------------------------------------+----------+
|verificacao                                                      |quantidade|
+-----------------------------------------------------------------+----------+
|Idade negativa                                                   |0         |
|Idade igual a zero — verificar contexto                          |603       |
|Idade acima de 110 — verificar                                   |432       |
|Ano de fabricação igual a zero                                   |29808     |
|Ano de fabricação negativo                                       |0         |
|Ano de fabricação posterior ao acidente — verificar              |0         |
|Quilometragem negativa                                           |0         |
|Quilometragem igual a zero — verificar                           |1253      |
|Quilometragem acima de 1000 km — revisar, sem ex

#### 3.6 — Repetição dos identificadores

Verificamos a frequência de `id`, `pesid` e `id_veiculo`, separando os marcadores ausentes. O propósito é identificar a **granularidade** da tabela: um `id` repetido não prova duplicação incorreta. Nenhum identificador artificial é criado, e as decisões de agrupamento serão tomadas somente após verificar a consistência dos atributos.

In [8]:
# 3.6 — Identificadores e repetição de IDs
# IDs repetidos podem ser esperados quando um acidente tem várias linhas associadas.
# Valores ausentes (inclusive strings vazias) são excluídos das contagens de IDs.
for coluna in ["id", "pesid", "id_veiculo"]:
    if coluna not in df_original.columns:
        continue

    ausentes = df_original.filter(campo_ausente(coluna)).count()
    frequencias = (
        df_original.filter(~campo_ausente(coluna))
        .groupBy(F.trim(F.col(coluna).cast("string")).alias("identificador"))
        .count()
    )
    resumo_ids = frequencias.agg(
        F.count("*").alias("identificadores_distintos_validos"),
        F.count(F.when(F.col("count") > 1, 1)).alias("ids_com_mais_de_uma_linha"),
        F.max("count").alias("maximo_linhas_por_id")
    )
    print(f"\nIdentificador: {coluna}")
    print(f"Registros com identificador ausente/vazio: {ausentes:,}")
    resumo_ids.show(1, truncate=False)
    print("10 identificadores válidos com maior número de linhas (para inspeção):")
    frequencias.orderBy(F.desc("count"), F.asc("identificador")).show(10, truncate=False)

# Síntese dos principais indicadores da exploração
print("\nResumo geral da exploração:")
print(f"Registros: {total_registros:,}")
print(f"Colunas: {total_colunas}")
print(f"Duplicatas exatas: {total_duplicados:,}")
print(f"Colunas com nulos reais: {df_nulos.filter(F.col('quantidade_nula') > 0).count()}")
print(f"Colunas com falhas de conversão: {df_conversao.filter(F.col('falhas_conversao') > 0).count()}")



Identificador: id
Registros com identificador ausente/vazio: 0
+---------------------------------+-------------------------+--------------------+
|identificadores_distintos_validos|ids_com_mais_de_uma_linha|maximo_linhas_por_id|
+---------------------------------+-------------------------+--------------------+
|72529                            |65382                    |792                 |
+---------------------------------+-------------------------+--------------------+

10 identificadores válidos com maior número de linhas (para inspeção):
+-------------+-----+
|identificador|count|
+-------------+-----+
|653691       |792  |
|755105       |774  |
|691021       |630  |
|653066       |594  |
|717394       |585  |
|734818       |584  |
|661616       |576  |
|697817       |576  |
|730170       |560  |
|706383       |558  |
+-------------+-----+
only showing top 10 rows

Identificador: pesid
Registros com identificador ausente/vazio: 56,801
+---------------------------------+---------

### 3.6.1 — Investigação das repetições do identificador de acidente

A repetição de `id` exige distinguir uma ocorrência de suas linhas associadas a pessoas, veículos, causas e tipos. O propósito é examinar a granularidade **sem excluir linhas**. Uma contagem de linhas não equivale ao número de acidentes distintos.

O diagnóstico abaixo mede quantas linhas cada `id` ocupa e se há variação nos atributos vinculados. Essas informações ajudam a decidir se uma chave composta é necessária para identificar linhas sem tratar repetições legítimas como duplicatas.

In [9]:
# 3.6.1 — Investigação aprofundada da repetição do id
# Esta célula é diagnóstica: não modifica df_original nem remove registros.

if "id" not in df_original.columns:
    print("A coluna 'id' não existe na base; não foi possível realizar a análise.")
else:
    # Excluímos IDs ausentes/vazios da análise de frequência.
    df_ids_validos = df_original.filter(~campo_ausente("id")).withColumn(
        "_id_analise", F.trim(F.col("id").cast("string"))
    )

    # Quantidade de linhas associadas a cada ID.
    por_id = df_ids_validos.groupBy("_id_analise").agg(
        F.count("*").alias("linhas_por_id")
    )

    resumo = por_id.agg(
        F.count("*").alias("ids_distintos"),
        F.sum(F.when(F.col("linhas_por_id") > 1, 1).otherwise(0)).alias("ids_com_repeticao"),
        F.max("linhas_por_id").alias("maximo_linhas_por_id")
    ).first()

    linhas_ids_repetidos = (
        por_id.filter(F.col("linhas_por_id") > 1)
        .agg(F.sum("linhas_por_id").alias("linhas_associadas_a_ids_repetidos"))
        .first()[0]
    ) or 0

    linhas_com_id_valido = df_ids_validos.count()

    print("Resumo da repetição de id")
    print(f"IDs distintos válidos: {resumo['ids_distintos']:,}")
    print(f"IDs associados a mais de uma linha: {resumo['ids_com_repeticao']:,}")
    print(f"Maior quantidade de linhas para um único ID: {resumo['maximo_linhas_por_id']:,}")
    print(
        "Linhas pertencentes a IDs repetidos: "
        f"{linhas_ids_repetidos:,} "
        f"({percentual(linhas_ids_repetidos, linhas_com_id_valido)}% das linhas com ID válido)"
    )

    print("\nDistribuição: quantos IDs aparecem 1, 2, 3... vezes")
    (
        por_id.groupBy("linhas_por_id")
        .agg(F.count("*").alias("quantidade_de_ids"))
        .orderBy("linhas_por_id")
        .show(20, truncate=False)
    )

    # Examina se os IDs mais repetidos estão associados a diferentes envolvidos,
    # veículos ou tipos de envolvido. Só utiliza colunas existentes na fonte.
    campos_contexto = [
        c for c in [
            "pesid", "id_veiculo", "tipo_envolvido",
            "data_inversa", "uf", "municipio", "tipo_acidente"
        ] if c in df_original.columns
    ]

    agregacoes = [F.count("*").alias("linhas_por_id")]
    for campo in campos_contexto:
        agregacoes.append(
            F.countDistinct(F.when(~campo_ausente(campo), F.trim(F.col(campo).cast("string"))))
             .alias(f"{campo}_distintos")
        )

    detalhe_ids = (
        df_ids_validos.groupBy("_id_analise")
        .agg(*agregacoes)
        .orderBy(F.desc("linhas_por_id"), F.asc("_id_analise"))
    )

    print("\n15 IDs com mais linhas e diversidade dos campos associados")
    detalhe_ids.show(15, truncate=False)

    # Exibe registros reais de alguns IDs mais repetidos para permitir conferência
    # dos campos disponíveis, sem imprimir todas as colunas da base.
    ids_exemplo = [
        linha["_id_analise"]
        for linha in detalhe_ids.select("_id_analise").limit(5).collect()
    ]

    colunas_exemplo = [
        c for c in [
            "id", "pesid", "id_veiculo", "tipo_envolvido",
            "data_inversa", "uf", "municipio", "tipo_acidente"
        ] if c in df_original.columns
    ]

    if ids_exemplo and colunas_exemplo:
        print("\nExemplos de registros para os 5 IDs mais repetidos")
        (
            df_ids_validos
            .filter(F.col("_id_analise").isin(ids_exemplo))
            .select(*[F.col(c) for c in colunas_exemplo])
            .orderBy(F.col("id").cast("string"))
            .show(50, truncate=False)
        )

    print(
        "\nInterpretação: a repetição de 'id' pode ser esperada quando um acidente "
        "possui várias linhas associadas. Compare a quantidade de linhas com a "
        "diversidade de envolvidos, veículos e demais campos. Não classifique "
        "esses casos como duplicatas sem verificar o significado das colunas."
    )


Resumo da repetição de id
IDs distintos válidos: 72,529
IDs associados a mais de uma linha: 65,382
Maior quantidade de linhas para um único ID: 792
Linhas pertencentes a IDs repetidos: 576,863 (98.78% das linhas com ID válido)

Distribuição: quantos IDs aparecem 1, 2, 3... vezes
+-------------+-----------------+
|linhas_por_id|quantidade_de_ids|
+-------------+-----------------+
|1            |7147             |
|2            |17143            |
|3            |6999             |
|4            |10126            |
|5            |1278             |
|6            |8458             |
|7            |287              |
|8            |3706             |
|9            |2287             |
|10           |998              |
|11           |26               |
|12           |4552             |
|13           |8                |
|14           |225              |
|15           |520              |
|16           |1040             |
|17           |6                |
|18           |2100             |
|19   

#### 3.6.1.1 — Diferentes tipos registrados por acidente e chave composta

A verificação conta os tipos distintos de `tipo_acidente` por `id`, observa a maior `ordem_tipo_acidente` e testa repetição de uma chave composta. A ordem indica a posição do tipo no registro, **não uma escala de gravidade**. Os resultados servem para reconhecer a granularidade; não representam número de acidentes de cada tipo nem justificam eliminar linhas. A ausência de repetições da chave examinada não prova unicidade para toda possível combinação de atributos.

In [10]:
# Diagnósticos de granularidade: não realizam deduplicação.
por_acidente = df_original.filter(~campo_ausente("id")).groupBy("id").agg(
    F.countDistinct("tipo_acidente").alias("tipos_distintos"),
    F.max(F.expr("try_cast(`ordem_tipo_acidente` AS INT)")).alias("maior_ordem_tipo")
)
res = por_acidente.agg(
    F.count("*").alias("acidentes_distintos"),
    F.sum(F.when(F.col("tipos_distintos") > 1, 1).otherwise(0)).alias("acidentes_com_multiplos_tipos"),
    F.max("maior_ordem_tipo").alias("maior_ordem_encontrada")
).first()
print("Acidentes distintos:", res["acidentes_distintos"])
print("Acidentes com mais de um tipo:", res["acidentes_com_multiplos_tipos"])
print("Maior ordem_tipo_acidente encontrada:", res["maior_ordem_encontrada"])
chave_composta = ["id", "pesid", "id_veiculo", "causa_acidente", "ordem_tipo_acidente"]
repeticoes_chave = df_original.groupBy(*chave_composta).count().filter(F.col("count") > 1).count()
print("Combinações repetidas de id+pesid+id_veiculo+causa_acidente+ordem_tipo_acidente:", repeticoes_chave)
print("Conclusão operacional: repetições de id não são, por si só, duplicatas exatas.")

Acidentes distintos: 72529
Acidentes com mais de um tipo: 33348
Maior ordem_tipo_acidente encontrada: 14
Combinações repetidas de id+pesid+id_veiculo+causa_acidente+ordem_tipo_acidente: 0
Conclusão operacional: repetições de id não são, por si só, duplicatas exatas.


#### 3.6.2 — Por que uma pessoa pode ocupar várias linhas

Analisamos as frequências de `pesid` e amostras dos registros associados a uma mesma pessoa. O objetivo é demonstrar por que **somar linhas não equivale a contar pessoas**. A investigação ajuda a decidir como construir, mais tarde, uma visão com uma linha por pessoa identificada.

In [11]:
# 3.6.2 — Investigação aprofundada da repetição do pesid
# Esta célula é diagnóstica: não modifica a base original.

if "pesid" not in df_original.columns:
    print("A coluna 'pesid' não existe na base; não foi possível realizar a análise.")

else:
    # Excluir identificadores ausentes ou vazios da análise de frequência.
    df_pesid_validos = (
        df_original
        .filter(~campo_ausente("pesid"))
        .withColumn(
            "_pesid_analise",
            F.trim(F.col("pesid").cast("string"))
        )
    )

    # Contar registros associados a cada pesid.
    frequencia_pesid = (
        df_pesid_validos
        .groupBy("_pesid_analise")
        .agg(
            F.count("*").alias("quantidade_registros")
        )
    )

    # Mostrar os 20 pesid com maior número de registros.
    print("20 pesid com maior quantidade de registros:")

    (
        frequencia_pesid
        .orderBy(
            F.desc("quantidade_registros"),
            F.asc("_pesid_analise")
        )
        .show(20, truncate=False)
    )

    # Selecionar os cinco pesid mais repetidos.
    pesid_mais_repetidos = (
        frequencia_pesid
        .orderBy(
            F.desc("quantidade_registros"),
            F.asc("_pesid_analise")
        )
        .limit(5)
        .select("_pesid_analise")
    )

    # Recuperar todas as colunas originais desses registros.
    print("\nRegistros completos dos cinco pesid mais repetidos:")

    (
        df_pesid_validos
        .join(
            pesid_mais_repetidos,
            on="_pesid_analise",
            how="inner"
        )
        .drop("_pesid_analise")
        .orderBy("pesid")
        .show(100, truncate=False)
    )

20 pesid com maior quantidade de registros:
+--------------+--------------------+
|_pesid_analise|quantidade_registros|
+--------------+--------------------+
|1621372       |57                  |
|1621373       |57                  |
|1621374       |57                  |
|1631792       |56                  |
|1632046       |56                  |
|1632048       |56                  |
|1632053       |56                  |
|1632055       |56                  |
|1632069       |56                  |
|1536361       |48                  |
|1536362       |48                  |
|1552469       |48                  |
|1552477       |48                  |
|1552478       |48                  |
|1627920       |48                  |
|1515432       |45                  |
|1517294       |45                  |
|1517296       |45                  |
|1517298       |45                  |
|1517299       |45                  |
+--------------+--------------------+
only showing top 20 rows

Registros completo

### 3.7 — Ausências segundo o tipo de envolvimento

A ausência de campos de pessoa e veículo pode depender do tipo de envolvimento. É importante distinguir campos não pertinentes, como informações do próprio veículo associadas a pedestres, de informações que seriam pertinentes mas estão desconhecidas. A avaliação é diagnóstica: nesta seção, nenhum campo é preenchido ou apagado.

O código calcula, para cada categoria de `tipo_envolvido`, a disponibilidade dos campos originais. A tabela integral permanece acessível como `df_ausencias_tipo_todas_colunas`; exibir apenas um resumo evita outputs extensos e repetitivos.

In [12]:
# 3.7 — Relação entre valores ausentes e tipo_envolvido: TODAS as colunas
# Calculamos ausências e preenchimentos para todas as colunas, mas exibimos
# apenas os resultados mais relevantes para facilitar a leitura do notebook.
if "tipo_envolvido" not in df_original.columns:
    print("A coluna tipo_envolvido não existe nesta versão da base; análise não executada.")
else:
    df_tipo_base = df_original.withColumn(
        "_grupo_tipo_envolvido",
        F.when(campo_ausente("tipo_envolvido"), F.lit("[AUSENTE]"))
         .otherwise(F.trim(F.col("tipo_envolvido")))
    )

    # Uma agregação por tipo_envolvido conta ausentes e preenchidos para cada coluna.
    expressoes = [F.count(F.lit(1)).alias("total_registros")]
    for i, coluna in enumerate(df_original.columns):
        expressoes.extend([
            F.sum(F.when(campo_ausente(coluna), 1).otherwise(0)).alias(f"aus_{i}"),
            F.sum(F.when(~campo_ausente(coluna), 1).otherwise(0)).alias(f"pre_{i}")
        ])
    agregados_tipo = df_tipo_base.groupBy("_grupo_tipo_envolvido").agg(*expressoes).collect()

    linhas_ausencias = []
    for registro in agregados_tipo:
        grupo = registro["_grupo_tipo_envolvido"]
        total_grupo = int(registro["total_registros"] or 0)
        for i, coluna in enumerate(df_original.columns):
            ausentes = int(registro[f"aus_{i}"] or 0)
            preenchidos = int(registro[f"pre_{i}"] or 0)
            linhas_ausencias.append((
                grupo, coluna, total_grupo, ausentes, preenchidos,
                percentual(ausentes, total_grupo)
            ))

    df_ausencias_tipo_todas_colunas = spark.createDataFrame(
        linhas_ausencias,
        "tipo_envolvido string, coluna_analisada string, total_registros long, "
        "quantidade_ausente long, quantidade_preenchida long, percentual_ausente_no_tipo double"
    )

    # Exibição resumida: 10 colunas com maior percentual de ausência em cada grupo.
    from pyspark.sql import Window
    janela_tipo = Window.partitionBy("tipo_envolvido").orderBy(
        F.desc("percentual_ausente_no_tipo"),
        F.desc("quantidade_ausente"),
        F.asc("coluna_analisada")
    )
    df_resumo_ausencias_tipo = (
        df_ausencias_tipo_todas_colunas
        .withColumn("posicao", F.row_number().over(janela_tipo))
        .filter(F.col("posicao") <= 10)
        .orderBy("tipo_envolvido", "posicao")
    )
    print("10 colunas com maior percentual de ausência em cada tipo_envolvido:")
    df_resumo_ausencias_tipo.show(max(df_resumo_ausencias_tipo.count(), 1), truncate=False)
    print("DataFrame completo disponível em: df_ausencias_tipo_todas_colunas")

    # Resumo de todas as colunas nos registros sem tipo_envolvido.
    sem_tipo = campo_ausente("tipo_envolvido")
    expressoes_sem_tipo = []
    for i, coluna in enumerate(df_original.columns):
        expressoes_sem_tipo.extend([
            F.sum(F.when(campo_ausente(coluna), 1).otherwise(0)).alias(f"aus_{i}"),
            F.sum(F.when(~campo_ausente(coluna), 1).otherwise(0)).alias(f"pre_{i}")
        ])
    registro_sem_tipo = df_original.filter(sem_tipo).agg(*expressoes_sem_tipo).first()
    total_sem_tipo = df_original.filter(sem_tipo).count()
    print(f"\nRegistros com tipo_envolvido ausente: {total_sem_tipo:,}")
    if total_sem_tipo > 0:
        linhas_sem_tipo = []
        for i, coluna in enumerate(df_original.columns):
            ausentes = int(registro_sem_tipo[f"aus_{i}"] or 0)
            preenchidos = int(registro_sem_tipo[f"pre_{i}"] or 0)
            linhas_sem_tipo.append((
                coluna, ausentes, preenchidos,
                percentual(ausentes, total_sem_tipo), percentual(preenchidos, total_sem_tipo)
            ))
        df_sem_tipo_todas_colunas = spark.createDataFrame(
            linhas_sem_tipo,
            "coluna string, ausentes long, preenchidos long, percentual_ausente double, percentual_preenchido double"
        )
        print("15 colunas com maior percentual de ausência entre registros sem tipo_envolvido:")
        df_sem_tipo_todas_colunas.orderBy(
            F.desc("percentual_ausente"), F.desc("ausentes"), F.asc("coluna")
        ).show(15, truncate=False)
        print("DataFrame completo disponível em: df_sem_tipo_todas_colunas")
    else:
        print("Não há registros com tipo_envolvido ausente; resumo específico não se aplica.")


10 colunas com maior percentual de ausência em cada tipo_envolvido:
+--------------+----------------------+---------------+------------------+---------------------+--------------------------+-------+
|tipo_envolvido|coluna_analisada      |total_registros|quantidade_ausente|quantidade_preenchida|percentual_ausente_no_tipo|posicao|
+--------------+----------------------+---------------+------------------+---------------------+--------------------------+-------+
|Cavaleiro     |ano_fabricacao_veiculo|100            |100               |0                    |100.0                     |1      |
|Cavaleiro     |id_veiculo            |100            |100               |0                    |100.0                     |2      |
|Cavaleiro     |marca                 |100            |100               |0                    |100.0                     |3      |
|Cavaleiro     |tipo_veiculo          |100            |100               |0                    |100.0                     |4      |
|Cavalei

### 3.8 — Investigação de idade igual a zero

A idade igual a zero não deve ser automaticamente classificada como erro. Ela pode representar uma criança com menos de um ano de idade. Para avaliar essa hipótese, cruzamos os registros com `tipo_envolvido`, `estado_fisico` e `sexo`.

No arquivo analisado, há **603 registros com idade igual a zero**: **602 são passageiros e 1 é pedestre**; não há condutores nesse grupo. Esse padrão é compatível com a possibilidade de serem bebês, mas o valor zero, isoladamente, não comprova a idade exata nem autoriza uma correção automática.


**Detalhamento do diagnóstico de idade zero**

Verificamos os registros com `idade = 0` e sua distribuição por outros atributos. Zero pode indicar idade inferior a um ano e, por isso, **não o transformamos automaticamente em ausência**. Essa decisão evita perder informação válida ao impor uma regra incorreta de idade mínima.

In [13]:
idade_num = F.expr("try_cast(replace(trim(`idade`), ',', '.') AS DOUBLE)")
idade_zero = df_original.filter(~campo_ausente("idade") & (idade_num == 0))

print("Quantidade de registros com idade igual a zero:")
print(f"Total de registros com idade zero: {idade_zero.count():,}")

# Reutiliza a mesma lógica para manter as três distribuições completas.
def mostrar_distribuicao_idade_zero(coluna):
    agrupamento = F.when(campo_ausente(coluna), F.lit("[AUSENTE]")).otherwise(
        F.trim(F.col(coluna))
    ).alias(coluna)
    print(f"Distribuição por {coluna}:")
    distribuicao = idade_zero.groupBy(agrupamento).count().orderBy(F.desc("count"))
    distribuicao.show(max(distribuicao.count(), 1), truncate=False)

for coluna in ["tipo_envolvido", "estado_fisico", "sexo"]:
    mostrar_distribuicao_idade_zero(coluna)

print("Exemplos para inspeção contextual:")
idade_zero.select(
    "id", "tipo_envolvido", "estado_fisico", "sexo", "tipo_veiculo", "idade"
).show(20, truncate=False)


Quantidade de registros com idade igual a zero:
Total de registros com idade zero: 603
Distribuição por tipo_envolvido:
+--------------+-----+
|tipo_envolvido|count|
+--------------+-----+
|Passageiro    |602  |
|Pedestre      |1    |
+--------------+-----+

Distribuição por estado_fisico:
+-------------+-----+
|estado_fisico|count|
+-------------+-----+
|Ileso        |250  |
|Lesões Leves |204  |
|Lesões Graves|118  |
|Óbito        |31   |
+-------------+-----+

Distribuição por sexo:
+---------+-----+
|sexo     |count|
+---------+-----+
|Masculino|316  |
|Feminino |287  |
+---------+-----+

Exemplos para inspeção contextual:
+------+--------------+-------------+---------+------------+-----+
|id    |tipo_envolvido|estado_fisico|sexo     |tipo_veiculo|idade|
+------+--------------+-------------+---------+------------+-----+
|663148|Passageiro    |Lesões Graves|Feminino |Automóvel   |0    |
|663718|Passageiro    |Lesões Graves|Feminino |Caminhonete |0    |
|664797|Passageiro    |Lesões 

### 3.9 — Anos de fabricação inferiores a 1900

A verificação investiga registros com ano de fabricação inferior a 1900; todos esses valores são iguais a **zero**. Portanto, não se trata de uma variedade de anos antigos, mas de um valor codificado ou inválido para o significado esperado da variável.

O zero deve ser investigado como possível marcador de informação ausente ou não informada. Como não temos a regra de negócio que originou o campo, não devemos afirmar que ele significa ausência com certeza nem substituí-lo silenciosamente. Na etapa de limpeza, a equipe deverá documentar a decisão adotada.

**Inspeção dos anos de fabricação antigos**

Isolamos valores de fabricação anteriores a 1900 para verificar frequência e contexto. Um ano antigo pode ser suspeito, mas a simples antiguidade não é motivo suficiente para invalidá-lo sem outra evidência. A limpeza posterior trata explicitamente anos menores ou iguais a zero.

In [14]:

ano_fabricacao_num = F.expr(
    "try_cast(replace(trim(`ano_fabricacao_veiculo`), ',', '.') AS DOUBLE)"
)

anos_abaixo_1900 = df_original.filter(
    ~campo_ausente("ano_fabricacao_veiculo")
    & (ano_fabricacao_num < 1900)
)

print("Quantidade de anos de fabricação inferiores a 1900:")
print(f"Total de registros com ano anterior a 1900: {anos_abaixo_1900.count():,}")

print("Valores encontrados abaixo de 1900:")
distribuicao_anos_suspeitos = anos_abaixo_1900.groupBy(
    ano_fabricacao_num.alias("ano_fabricacao")
).count().orderBy("ano_fabricacao")
distribuicao_anos_suspeitos.show(max(distribuicao_anos_suspeitos.count(), 1), truncate=False)

print("Distribuição dos anos suspeitos por tipo de veículo:")
distribuicao_por_veiculo = anos_abaixo_1900.groupBy(
    F.when(campo_ausente("tipo_veiculo"), F.lit("[AUSENTE]"))
     .otherwise(F.trim(F.col("tipo_veiculo"))).alias("tipo_veiculo"),
    ano_fabricacao_num.alias("ano_fabricacao")
).count().orderBy(F.desc("count"))
distribuicao_por_veiculo.show(max(distribuicao_por_veiculo.count(), 1), truncate=False)

# Checagem explícita: existem valores abaixo de 1900 diferentes de zero?
anos_abaixo_zero = anos_abaixo_1900.filter(ano_fabricacao_num != 0)
print(
    "Registros abaixo de 1900 com valor diferente de zero:",
    anos_abaixo_zero.count()
)


Quantidade de anos de fabricação inferiores a 1900:
Total de registros com ano anterior a 1900: 29,808
Valores encontrados abaixo de 1900:
+--------------+-----+
|ano_fabricacao|count|
+--------------+-----+
|0.0           |29808|
+--------------+-----+

Distribuição dos anos suspeitos por tipo de veículo:
+-----------------+--------------+-----+
|tipo_veiculo     |ano_fabricacao|count|
+-----------------+--------------+-----+
|Motocicleta      |0.0           |7870 |
|Automóvel        |0.0           |7231 |
|Bicicleta        |0.0           |4959 |
|Caminhão         |0.0           |3311 |
|Caminhão-trator  |0.0           |1432 |
|Caminhonete      |0.0           |1136 |
|Outros           |0.0           |942  |
|Semireboque      |0.0           |842  |
|Motoneta         |0.0           |527  |
|Ciclomotor       |0.0           |314  |
|Camioneta        |0.0           |285  |
|Ônibus           |0.0           |202  |
|Reboque          |0.0           |190  |
|Carroça-charrete |0.0           |14

### 3.10 — Investigação dos valores extremos

O intervalo interquartil (IQR) é empregado **como sinalizador exploratório** para idade e ano de fabricação, nunca como regra automática de exclusão. A distribuição dessas variáveis pode conter valores raros legítimos; limites estatísticos não são sinônimo de impossibilidade. Em particular, não se aplica IQR às flags binárias de vítimas nem ao `km` da rodovia: este último é a **posição quilométrica** onde ocorreu o acidente, não uma intensidade de risco ou gravidade.

A decisão final diferencia **valor implausível**, submetido a uma regra de invalidação fundamentada, de **valor extremo plausível**, mantido na base.

**Método exploratório do IQR**

O resumo identifica observações que merecem investigação, sem remover registros e sem transformar marcadores de ausência em números artificiais. O limite de 110 anos decorre da inspeção de idades incompatíveis, incluindo anos como `2024` e `1924` inseridos no campo de idade; essa regra é uma escolha conservadora e documentada.

In [15]:
# 3.10 — IQR e possíveis outliers
# Requer df_numerico e as colunas *_num criadas na seção 3.4.

variaveis_iqr = [
    "idade",
    "ano_fabricacao_veiculo",
]
variaveis_iqr = [
    c for c in variaveis_iqr
    if c in df_numerico.columns and f"{c}_num" in df_numerico.columns
]

resultados_iqr = []

for coluna in variaveis_iqr:
    coluna_num = f"{coluna}_num"
    dados_iqr = df_numerico.filter(F.col(coluna_num).isNotNull())

    # Os zeros do ano de fabricação já foram identificados como suspeitos.
    # Para calcular quartis de anos reais, consideramos apenas anos > 0.
    if coluna == "ano_fabricacao_veiculo":
        dados_iqr = dados_iqr.filter(F.col(coluna_num) > 0)

    quantidade_valida = dados_iqr.count()
    if quantidade_valida == 0:
        continue

    q1, q3 = dados_iqr.approxQuantile(coluna_num, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr

    condicao_outlier = (
        (F.col(coluna_num) < F.lit(limite_inferior))
        | (F.col(coluna_num) > F.lit(limite_superior))
    )

    resumo = dados_iqr.agg(
        F.count(F.when(condicao_outlier, 1)).alias("quantidade_outliers"),
        F.count(F.when(F.col(coluna_num) < limite_inferior, 1)).alias("abaixo_limite"),
        F.count(F.when(F.col(coluna_num) > limite_superior, 1)).alias("acima_limite"),
        F.min(F.when(condicao_outlier, F.col(coluna_num))).alias("menor_outlier"),
        F.max(F.when(condicao_outlier, F.col(coluna_num))).alias("maior_outlier")
    ).first()

    qtd_outliers = int(resumo["quantidade_outliers"] or 0)
    resultados_iqr.append((
        coluna,
        float(q1),
        float(q3),
        float(iqr),
        float(limite_inferior),
        float(limite_superior),
        int(quantidade_valida),
        qtd_outliers,
        round(qtd_outliers / quantidade_valida * 100, 2),
        int(resumo["abaixo_limite"] or 0),
        int(resumo["acima_limite"] or 0),
        resumo["menor_outlier"],
        resumo["maior_outlier"]
    ))

schema_iqr = """
    variavel string,
    Q1 double,
    Q3 double,
    IQR double,
    limite_inferior double,
    limite_superior double,
    quantidade_valida long,
    quantidade_outliers long,
    percentual_outliers double,
    abaixo_limite long,
    acima_limite long,
    menor_outlier double,
    maior_outlier double
"""

df_resultados_iqr = spark.createDataFrame(resultados_iqr, schema_iqr)

print("Resumo de possíveis outliers pelo método IQR:")
df_resultados_iqr.orderBy(
    F.desc("percentual_outliers")
).show(len(resultados_iqr), truncate=False)


Resumo de possíveis outliers pelo método IQR:
+----------------------+------+------+----+---------------+---------------+-----------------+-------------------+-------------------+-------------+------------+-------------+-------------+
|variavel              |Q1    |Q3    |IQR |limite_inferior|limite_superior|quantidade_valida|quantidade_outliers|percentual_outliers|abaixo_limite|acima_limite|menor_outlier|maior_outlier|
+----------------------+------+------+----+---------------+---------------+-----------------+-------------------+-------------------+-------------+------------+-------------+-------------+
|ano_fabricacao_veiculo|2010.0|2021.0|11.0|1993.5         |2037.5         |535585           |9631               |1.8                |9631         |0           |1900.0       |1993.0       |
|idade                 |28.0  |50.0  |22.0|-5.0           |83.0           |478397           |1521               |0.32               |0            |1521        |84.0         |2024.0       |
+--------

#### 3.10.1 — Exemplos que fundamentam a regra de plausibilidade etária

O limite de 110 anos não é deduzido do IQR: ele é uma **regra conservadora de plausibilidade** adotada para identificar erros evidentes de preenchimento. Valores como 125, 225, 914 e anos de calendário no campo `idade` justificam a investigação. A decisão de substituir valores superiores a 110 por `NULL` é documentada na limpeza; extremos ainda plausíveis são mantidos, sem remoção de linhas.

In [16]:
# Evidência concreta para justificar a decisão sobre idade, antes da limpeza.
idades_extremas = df_original.filter(
    ~campo_ausente("idade") &
    (F.expr("try_cast(replace(trim(`idade`), ',', '.') AS DOUBLE)") > 110)
)
print("Idades superiores a 110:", idades_extremas.count())
idades_extremas.groupBy("idade").count().orderBy(F.desc("count")).show(12, truncate=False)
print("Distribuição por tipo_envolvido:")
idades_extremas.groupBy("tipo_envolvido").count().orderBy(F.desc("count")).show(12, truncate=False)

Idades superiores a 110: 432
+-----+-----+
|idade|count|
+-----+-----+
|125  |142  |
|2024 |98   |
|914  |71   |
|225  |47   |
|1924 |35   |
|1914 |9    |
|1024 |8    |
|1025 |7    |
|327  |4    |
|1046 |4    |
|1028 |2    |
|127  |2    |
+-----+-----+
only showing top 12 rows
Distribuição por tipo_envolvido:
+--------------+-----+
|tipo_envolvido|count|
+--------------+-----+
|Passageiro    |395  |
|Condutor      |25   |
|Pedestre      |12   |
+--------------+-----+



**Decisão de tratamento de extremos.** Não se remove nenhuma linha somente por estar fora dos limites do IQR: isso descartaria acidentes reais potencialmente raros e distorceria a distribuição. Em contrapartida, idades negativas ou superiores a 110 anos são consideradas incompatíveis com a finalidade do campo. Exemplos observados no arquivo, como 125, 914, 1924 e 2024, reforçam a hipótese de erro de registro, mas não autorizam inventar uma idade substituta. O valor passa a `NULL`, e a linha é mantida. Idade igual a zero permanece válida como possível representação de pessoa com menos de um ano. Ano de fabricação igual a zero não é um ano válido e também passa a `NULL`.

#### 3.11 — Relação entre pessoas não identificadas e dados do acidente

Investigamos a coincidência entre falta de `pesid` e de `tipo_envolvido`, bem como a disponibilidade de outros dados nas mesmas linhas. Um registro sem pessoa identificada pode continuar útil para caracterizar um acidente. Não preenchemos IDs por suposição nem excluímos essas linhas; as restrições de análises por pessoa devem ser documentadas.

In [17]:
# 3.11 — Análise de registros sem identificação de pessoa

total = df_original.count()

# Identificação dos registros com informações ausentes
sem_pesid = campo_ausente("pesid")
sem_tipo = campo_ausente("tipo_envolvido")

df_sem_pesid = df_original.filter(sem_pesid)

# Contagens
qtd_sem_pesid = df_sem_pesid.count()
qtd_sem_tipo = df_original.filter(sem_tipo).count()

qtd_sem_ambos = df_original.filter(
    sem_pesid & sem_tipo
).count()

qtd_sem_pesid_com_tipo = df_original.filter(
    sem_pesid & ~sem_tipo
).count()

# Resumo
dados_resumo = [
    ("Sem pesid", qtd_sem_pesid),
    ("Sem tipo_envolvido", qtd_sem_tipo),
    ("Sem ambos", qtd_sem_ambos),
    ("Sem pesid, mas com tipo conhecido",
     qtd_sem_pesid_com_tipo)
]

df_resumo = spark.createDataFrame(
    [
        (situacao, qtd, percentual(qtd, total))
        for situacao, qtd in dados_resumo
    ],
    "situacao string, registros long, percentual double"
)

print("Ausências relacionadas à identificação de pessoas:")
df_resumo.show(truncate=False)

# Verificar quais informações permanecem disponíveis
print("\nInformações disponíveis nos registros sem pesid:")

colunas_analise = [
    "id", "data_inversa", "uf", "municipio",
    "causa_acidente", "tipo_acidente",
    "id_veiculo", "idade", "sexo", "estado_fisico"
]

expressoes = [
    F.sum(
        F.when(~campo_ausente(c), 1).otherwise(0)
    ).alias(f"c{i}")
    for i, c in enumerate(colunas_analise)
]

resultado = df_sem_pesid.agg(*expressoes).first()

dados = [
    (
        c,
        int(resultado[f"c{i}"] or 0),
        percentual(
            int(resultado[f"c{i}"] or 0),
            qtd_sem_pesid
        )
    )
    for i, c in enumerate(colunas_analise)
]

df_disponibilidade = spark.createDataFrame(
    dados,
    "coluna string, preenchidos long, percentual_preenchido double"
)

df_disponibilidade.show(truncate=False)

print("\nExemplos de registros sem pesid:")

df_sem_pesid.select(
    "id", "pesid", "tipo_envolvido",
    "idade", "sexo", "id_veiculo",
    "causa_acidente"
).show(5, truncate=False)


Ausências relacionadas à identificação de pessoas:
+---------------------------------+---------+----------+
|situacao                         |registros|percentual|
+---------------------------------+---------+----------+
|Sem pesid                        |56801    |9.73      |
|Sem tipo_envolvido               |56801    |9.73      |
|Sem ambos                        |56801    |9.73      |
|Sem pesid, mas com tipo conhecido|0        |0.0       |
+---------------------------------+---------+----------+


Informações disponíveis nos registros sem pesid:
+--------------+-----------+---------------------+
|coluna        |preenchidos|percentual_preenchido|
+--------------+-----------+---------------------+
|id            |56801      |100.0                |
|data_inversa  |56801      |100.0                |
|uf            |56801      |100.0                |
|municipio     |56801      |100.0                |
|causa_acidente|56801      |100.0                |
|tipo_acidente |56801      |100.0 

#### 3.11.1 — Disponibilidade de idade entre registros de condutores

Para a futura pergunta sobre condutores menores de idade, calculamos a parcela de **linhas classificadas como condutor** em que a idade está ausente. Esse percentual indica cobertura do campo no conjunto de registros, **não a proporção de condutores distintos sem idade**, pois a mesma pessoa pode aparecer em mais de uma linha. Não haverá imputação de idade nem inferência de menoridade nesta etapa.

In [18]:
# Registros de condutores com idade ausente (métrica por linha).
condutores = df_original.filter(F.lower(F.trim(F.col("tipo_envolvido"))) == "condutor")
qtd_condutores = condutores.count()
qtd_sem_idade = condutores.filter(campo_ausente("idade")).count()
print(f"Registros de condutores: {qtd_condutores:,}")
print(f"Condutores sem idade informada: {qtd_sem_idade:,} ({percentual(qtd_sem_idade, qtd_condutores)}%)")
print("Não atribuímos idades nem removemos registros.")

Registros de condutores: 354,002
Condutores sem idade informada: 26,897 (7.6%)
Não atribuímos idades nem removemos registros.


### 3.12 — Cobertura da classificação do acidente

Esta verificação mede registros sem `classificacao_acidente` e mostra se há indicadores individuais de morte nas linhas afetadas. **Não se considera divergência** a coexistência de `mortos=0` e `mortos=1` em pessoas diferentes do mesmo acidente: cada indicador se refere à pessoa da própria linha. A falta de classificação pode limitar uma análise por gravidade, mas não autoriza reconstruir automaticamente o rótulo do acidente a partir de uma linha isolada. A informação permanece ausente e a limitação é documentada.

In [19]:
# 3.12 — Cobertura da classificação do acidente (indicadores por pessoa)
sem_classe = df_original.filter(campo_ausente("classificacao_acidente"))
qtd_sem_classe = sem_classe.count()
qtd_morto = sem_classe.filter(F.expr("try_cast(trim(`mortos`) AS INT)") == 1).count()
ids_afetados = sem_classe.select("id").distinct().count()
print(f"Linhas com classificacao_acidente ausente: {qtd_sem_classe:,}")
print(f"Acidentes distintos associados: {ids_afetados:,}")
print(f"Linhas de pessoas com indicador individual mortos=1: {qtd_morto:,}")
sem_classe.select("id", "pesid", "classificacao_acidente", "mortos").show(10, truncate=False)
print("Valores 0 e 1 em linhas diferentes são compatíveis com diferentes pessoas do mesmo acidente.")


Linhas com classificacao_acidente ausente: 7
Acidentes distintos associados: 1
Linhas de pessoas com indicador individual mortos=1: 1
+------+-------+----------------------+------+
|id    |pesid  |classificacao_acidente|mortos|
+------+-------+----------------------+------+
|652519|1478721|NA                    |0     |
|652519|1467833|NA                    |0     |
|652519|1467832|NA                    |1     |
|652519|1467829|NA                    |0     |
|652519|NA     |NA                    |NA    |
|652519|NA     |NA                    |NA    |
|652519|1470998|NA                    |0     |
+------+-------+----------------------+------+

Valores 0 e 1 em linhas diferentes são compatíveis com diferentes pessoas do mesmo acidente.


## 4. Limpeza e preparação dos dados

A limpeza converte os diagnósticos em regras reproduzíveis. O arquivo original permanece em `df_original`; as correções são aplicadas em `df_tratado`, mantendo todas as linhas. A ordem é importante: **primeiro** registrar quando certos atributos de veículo não se aplicam, **depois** normalizar as formas de ausência, **em seguida** invalidar valores comprovadamente incompatíveis e **finalmente** atribuir os tipos físicos corretos. Sem essa sequência, uma coluna numérica não poderia distinguir `NULL` por desconhecimento de `NULL` por não aplicabilidade.

Não há imputação artificial de `pesid`, `tipo_envolvido` ou idade. Isso evita inventar pessoas e categorias e preserva a utilidade dos registros de acidente incompletos.

### 4.1 — Registrar os casos não aplicáveis

Os quatro campos `id_veiculo`, `tipo_veiculo`, `marca` e `ano_fabricacao_veiculo` podem estar ausentes em linhas de **Pedestre, Cavaleiro ou Testemunha** porque o registro da pessoa não apresenta um veículo próprio ao qual esses atributos se apliquem. O indicador booleano é marcado como verdadeiro **somente se** o campo já estiver ausente e o tipo de envolvimento estiver nessas categorias; nunca se apaga um dado veicular conhecido. Cada atributo ganha seu próprio sinalizador porque diferentes campos podem ter motivos de ausência distintos. Isso permite excluir casos não aplicáveis dos percentuais de falta de informação sem tratar todos os `NULL` como equivalentes.

É uma regra de interpretação da estrutura dos registros, não uma afirmação de que a pessoa nunca esteve relacionada a veículo algum.

In [20]:
# 4.1 — Registrar a não aplicabilidade antes de normalizar os marcadores
# A regra é conservadora: só sinaliza campos AUSENTES para Pedestre/Cavaleiro/Testemunha.
df_tratado = df_original
campos_veiculo = ["id_veiculo", "tipo_veiculo", "marca", "ano_fabricacao_veiculo"]
tipos_sem_veiculo_proprio = ["pedestre", "cavaleiro", "testemunha"]

condicao_tipo = F.lower(F.trim(F.col("tipo_envolvido"))).isin(tipos_sem_veiculo_proprio)
for c in campos_veiculo:
    df_tratado = df_tratado.withColumn(
        f"{c}_nao_se_aplica",
        (F.coalesce(condicao_tipo, F.lit(False)) & campo_ausente(c)).cast("boolean")
    )

for c in campos_veiculo:
    assert df_tratado.filter(F.col(c + "_nao_se_aplica") & ~campo_ausente(c)).count() == 0, c

print("Indicadores booleanos criados (True = ausência classificada como não aplicável):")
df_tratado.select(*[F.sum(F.col(c + "_nao_se_aplica").cast("int")).alias(c)
                    for c in campos_veiculo]).show(truncate=False)
print("Exemplos classificados:")
df_tratado.filter(F.col("id_veiculo_nao_se_aplica")).select(
    "tipo_envolvido", "id_veiculo", "id_veiculo_nao_se_aplica",
    "ano_fabricacao_veiculo", "ano_fabricacao_veiculo_nao_se_aplica"
).show(5, truncate=False)


Indicadores booleanos criados (True = ausência classificada como não aplicável):
+----------+------------+-----+----------------------+
|id_veiculo|tipo_veiculo|marca|ano_fabricacao_veiculo|
+----------+------------+-----+----------------------+
|18617     |18617       |18617|18617                 |
+----------+------------+-----+----------------------+

Exemplos classificados:
+--------------+----------+------------------------+----------------------+------------------------------------+
|tipo_envolvido|id_veiculo|id_veiculo_nao_se_aplica|ano_fabricacao_veiculo|ano_fabricacao_veiculo_nao_se_aplica|
+--------------+----------+------------------------+----------------------+------------------------------------+
|Testemunha    |NA        |true                    |NA                    |true                                |
|Testemunha    |NA        |true                    |NA                    |true                                |
|Testemunha    |NA        |true                    |NA

### 4.2 — Normalizar os marcadores de ausência para `NULL`

A exploração 3.2 mostrou que ausências aparecem com representações diferentes. Esta célula padroniza `NA`, strings vazias, `Ignorado`, `Não Informado` e variantes reconhecidas para o **`NULL` nativo do Spark**. Essa opção torna contagens e conversões consistentes e evita que as expressões tentem converter essas palavras para números. A informação **“não se aplica”** não se perde: já foi registrada nos indicadores da 4.1. Nenhuma categoria informativa é recodificada, e `tracado_via` permanece exatamente como está na fonte (à exceção da limpeza geral de espaços e marcadores ausentes).

In [21]:
# 4.2 — Substituir TODOS os marcadores de ausência por NULL real
# O motivo 'não se aplica' permanece somente nas colunas booleanas criadas em 4.1.
colunas_originais = list(df_original.columns)
expressoes = [F.sum(F.when(campo_ausente(c), 1).otherwise(0)).alias(f"n{i}")
              for i, c in enumerate(colunas_originais)]
antes = df_tratado.agg(*expressoes).first()

for c in colunas_originais:
    df_tratado = df_tratado.withColumn(
        c, F.when(campo_ausente(c), F.lit(None).cast("string"))
             .otherwise(F.trim(F.col(c).cast("string")))
    )

resumo = [(c, int(antes[f"n{i}"] or 0)) for i, c in enumerate(colunas_originais)]
spark.createDataFrame(resumo, "coluna string, marcadores_convertidos_para_null long")    .orderBy(F.desc("marcadores_convertidos_para_null")).show(len(resumo), False)
print("Exemplos após a normalização:")
df_tratado.filter(F.col("pesid").isNull()).select(
    "id", "pesid", "tipo_envolvido", "idade", "id_veiculo_nao_se_aplica"
).show(5, truncate=False)


+----------------------+--------------------------------+
|coluna                |marcadores_convertidos_para_null|
+----------------------+--------------------------------+
|idade                 |105613                          |
|sexo                  |93055                           |
|estado_fisico         |91884                           |
|pesid                 |56801                           |
|tipo_envolvido        |56801                           |
|ilesos                |56801                           |
|feridos_leves         |56801                           |
|feridos_graves        |56801                           |
|mortos                |56801                           |
|marca                 |41721                           |
|id_veiculo            |18617                           |
|tipo_veiculo          |18617                           |
|ano_fabricacao_veiculo|18617                           |
|condicao_metereologica|7130                            |
|br           

### 4.3 — Tratar valores manifestamente inválidos

A análise inicial encontrou idades superiores a 110 anos, incluindo registros como 125, 914, 1924 e 2024. O corte adotado é uma **regra conservadora e explícita de plausibilidade**, não um limite extraído do IQR. Valores abaixo de zero e acima de 110 tornam-se `NULL`; **idade zero é preservada**, pois pode corresponder a uma criança com menos de um ano. Para `ano_fabricacao_veiculo`, zero ou valores negativos não correspondem a um ano de fabricação e tornam-se `NULL`. Não se corrige automaticamente um número para outro, porque o valor verdadeiro é desconhecido. Cada transformação apresenta a quantidade afetada e mantém a linha original.

In [22]:
# 4.3 — Invalidar valores sem plausibilidade, mantendo NULL e registrando a regra
# Idade 0 é válida; ano de fabricação 0 é ausente/inválido.
def duplo(c):
    return F.expr(f"try_cast(replace(trim(`{c}`), ',', '.') AS DOUBLE)")

regras = {
    "idade": (duplo("idade") < 0) | (duplo("idade") > 110),
    "ano_fabricacao_veiculo": duplo("ano_fabricacao_veiculo") <= 0
}
for c, regra in regras.items():
    total = df_tratado.filter(regra).count()
    print(f"{c}: {total:,} valores inválidos convertidos em NULL")
    print("Exemplos antes:")
    df_tratado.filter(regra).select("id", c).show(3, False)
    df_tratado = df_tratado.withColumn(
        c, F.when(regra, F.lit(None).cast("string")).otherwise(F.col(c))
    )
    print("Verificação: quantidade total de NULL agora:", df_tratado.filter(F.col(c).isNull()).count())

idade: 432 valores inválidos convertidos em NULL
Exemplos antes:
+------+-----+
|id    |idade|
+------+-----+
|653206|2024 |
|653206|2024 |
|653211|2024 |
+------+-----+
only showing top 3 rows
Verificação: quantidade total de NULL agora: 106045
ano_fabricacao_veiculo: 29,808 valores inválidos convertidos em NULL
Exemplos antes:
+------+----------------------+
|id    |ano_fabricacao_veiculo|
+------+----------------------+
|652492|0                     |
|652492|0                     |
|652493|0                     |
+------+----------------------+
only showing top 3 rows
Verificação: quantidade total de NULL agora: 48425


### 4.4 — Diagnóstico preliminar das ausências

Após as substituições e regras de plausibilidade, o código separa **nulos físicos**, **nulos por não aplicabilidade** e **ausências em campos aplicáveis**. A taxa é calculada como `ausências aplicáveis / registros aplicáveis`, isto é, excluindo os não aplicáveis tanto do numerador quanto do denominador. Esse diagnóstico intermediário permite observar o efeito inicial da limpeza; o cálculo **definitivo** é repetido na 4.6, depois da conversão dos tipos, quando eventuais valores malformados também terão sido identificados.

In [23]:
# 4.4 — Nulos REAIS, não aplicáveis e percentual entre registros aplicáveis
# Os indicadores não são contabilizados como ausência.
total_tratado = df_tratado.count()
expressoes = [F.sum(F.col(c).isNull().cast("int")).alias(f"n{i}")
              for i, c in enumerate(colunas_originais)]
res = df_tratado.agg(*expressoes).first()
na = df_tratado.agg(*[
    F.sum(F.col(c + "_nao_se_aplica").cast("int")).alias(c)
    for c in campos_veiculo
]).first()
linhas = []
for i, c in enumerate(colunas_originais):
    nulos = int(res[f"n{i}"] or 0)
    nao_aplica = int(na[c] or 0) if c in campos_veiculo else 0
    assert nao_aplica <= nulos, f"Aplicabilidade inconsistente em {c}"
    nao_informados = nulos - nao_aplica
    aplicaveis = total_tratado - nao_aplica
    linhas.append((c, nulos, nao_informados, nao_aplica, aplicaveis,
                   round(100 * nao_informados / aplicaveis, 2) if aplicaveis else None))

df_resumo_ausencias = spark.createDataFrame(linhas,
    "coluna string, nulos_reais long, nao_informados long, nao_se_aplica long, registros_aplicaveis long, percentual_ausencia double")
print("Diagnóstico preliminar de ausência, antes da tipagem:")
df_resumo_ausencias.orderBy(F.desc_nulls_last("percentual_ausencia")).show(10, False)
print("Importante: NULL e 'não se aplica' não são conceitos equivalentes.")

Diagnóstico preliminar de ausência, antes da tipagem:
+----------------------+-----------+--------------+-------------+--------------------+-------------------+
|coluna                |nulos_reais|nao_informados|nao_se_aplica|registros_aplicaveis|percentual_ausencia|
+----------------------+-----------+--------------+-------------+--------------------+-------------------+
|idade                 |106045     |106045        |0            |584010              |18.16              |
|sexo                  |93055      |93055         |0            |584010              |15.93              |
|estado_fisico         |91884      |91884         |0            |584010              |15.73              |
|pesid                 |56801      |56801         |0            |584010              |9.73               |
|tipo_envolvido        |56801      |56801         |0            |584010              |9.73               |
|ilesos                |56801      |56801         |0            |584010              |9.73

### 4.5 — Atribuição dos tipos e auditoria de conversões

A tipagem só ocorre depois de substituir marcadores textuais por `NULL`. Idade, ano de fabricação, rodovia e flags individuais são convertidos para inteiro; coordenadas e quilometragem, para decimal; a data, para `DateType`. Identificadores (`id`, `pesid`, `id_veiculo`) permanecem strings para não alterar seu papel de chaves. `horario` é mantido como texto validado `HH:mm:ss`.

O uso de `try_cast` evita falha do processamento diante de valores malformados, **mas exige auditoria**: a célula conta e exibe os valores incompatíveis antes de convertê-los em `NULL`, compara a cobertura de valores conhecidos antes/depois e registra datas e horários inválidos. Assim, uma conversão silenciosa não é confundida com perda aceitável de dados.

In [24]:
# 4.5 — Converter tipos corretos sem introduzir texto em colunas numéricas
# try_cast não falha no modo ANSI; NULL continua NULL.
inteiros = ["idade", "ano_fabricacao_veiculo", "br", "ordem_tipo_acidente",
            "ilesos", "feridos_leves", "feridos_graves", "mortos"]
decimais = ["km", "latitude", "longitude"]

# Antes de converter, verificar valores conhecidos que não respeitam o tipo esperado.
problemas = []
for c in inteiros + decimais:
    n = duplo(c)
    invalido = (F.col(c).isNotNull() & (
        n.isNull() | F.isnan(n) | (F.abs(n) == float("inf"))
    ))
    if c in inteiros:
        invalido = invalido | (n.isNotNull() & (
            (n != F.floor(n)) | (n < -2147483648) | (n > 2147483647)))
    qtd = df_tratado.filter(invalido).count()
    problemas.append((c, qtd))
    if qtd:
        print(f"{c}: {qtd} valores incompatíveis, exemplos:")
        df_tratado.filter(invalido).select("id", c).show(3, False)
        # Valores malformados são inválidos, nunca 'não se aplica' automaticamente.
        df_tratado = df_tratado.withColumn(c, F.when(invalido, F.lit(None)).otherwise(F.col(c)))

for c in inteiros:
    antes_validos = df_tratado.filter(F.col(c).isNotNull()).count()
    df_tratado = df_tratado.withColumn(c, F.expr(
        f"try_cast(try_cast(replace(trim(`{c}`), ',', '.') AS DOUBLE) AS INT)"))
    depois_validos = df_tratado.filter(F.col(c).isNotNull()).count()
    assert antes_validos == depois_validos, f"Conversão INT gerou NULL não auditado em {c}"
for c in decimais:
    antes_validos = df_tratado.filter(F.col(c).isNotNull()).count()
    df_tratado = df_tratado.withColumn(c, F.expr(
        f"try_cast(replace(trim(`{c}`), ',', '.') AS DOUBLE)"))
    depois_validos = df_tratado.filter(F.col(c).isNotNull()).count()
    assert antes_validos == depois_validos, f"Conversão DOUBLE gerou NULL não auditado em {c}"

# Data no tipo DATE; horário mantido STRING ('HH:mm:ss') pela natureza do dado.
data_convertida = F.expr("try_cast(trim(`data_inversa`) AS DATE)")
qtd_data = df_tratado.filter(F.col("data_inversa").isNotNull() & data_convertida.isNull()).count()
df_tratado = df_tratado.withColumn("data_inversa", data_convertida)

h = F.trim(F.col("horario"))
valido_h = h.rlike(r"^(?:[01][0-9]|2[0-3]):[0-5][0-9](?::[0-5][0-9])?$")
qtd_horario = df_tratado.filter(h.isNotNull() & ~valido_h).count()
df_tratado = df_tratado.withColumn(
    "horario", F.when(h.isNull(), F.lit(None).cast("string"))
                 .when(~valido_h, F.lit(None).cast("string"))
                 .when(F.length(h) == 5, F.concat(h, F.lit(":00")))
                 .otherwise(h)
)
print("Falhas numéricas por coluna (convertidas em NULL):", problemas)
print("Datas inválidas:", qtd_data, "| Horários inválidos:", qtd_horario)
df_tratado.select("idade", "ano_fabricacao_veiculo", "km", "data_inversa", "horario").show(8, False)
df_tratado.printSchema()


Falhas numéricas por coluna (convertidas em NULL): [('idade', 0), ('ano_fabricacao_veiculo', 0), ('br', 0), ('ordem_tipo_acidente', 0), ('ilesos', 0), ('feridos_leves', 0), ('feridos_graves', 0), ('mortos', 0), ('km', 0), ('latitude', 0), ('longitude', 0)]
Datas inválidas: 0 | Horários inválidos: 0
+-----+----------------------+-----+------------+--------+
|idade|ano_fabricacao_veiculo|km   |data_inversa|horario |
+-----+----------------------+-----+------------+--------+
|NULL |2018                  |89.5 |2025-01-01  |00:30:00|
|28   |2009                  |89.5 |2025-01-01  |00:30:00|
|29   |2009                  |89.5 |2025-01-01  |00:30:00|
|NULL |NULL                  |89.5 |2025-01-01  |00:30:00|
|24   |2016                  |120.0|2025-01-01  |00:50:00|
|26   |2016                  |120.0|2025-01-01  |00:50:00|
|57   |2012                  |414.0|2025-01-01  |01:15:00|
|8    |2012                  |414.0|2025-01-01  |01:15:00|
+-----+----------------------+-----+------------+--

### 4.6 — Resultado final da limpeza

Esta é a conferência definitiva, feita **após todas as conversões**. O código testa a preservação do número de linhas, a presença das 37 colunas de origem e quatro sinalizadores, os tipos do esquema, a coerência dos indicadores (nenhum campo preenchido pode estar marcado como não aplicável) e o percentual final de ausência entre registros aos quais cada atributo se aplica. Os testes interrompem a execução diante de contradições; a tabela completa é mostrada uma única vez nesta etapa para evitar repetir uma saída extensa.

In [25]:
# 4.6 — Verificar integridade, tipos e percentuais FINAIS de ausência
from pyspark.sql.types import IntegerType, DoubleType, DateType, BooleanType, StringType

total_final = df_tratado.count()
assert total_final == df_original.count(), "Linhas foram perdidas"
assert df_tratado.columns[:len(colunas_originais)] == colunas_originais
assert len(df_tratado.columns) == len(colunas_originais) + len(campos_veiculo)
for c in inteiros:
    assert isinstance(df_tratado.schema[c].dataType, IntegerType), c
for c in decimais:
    assert isinstance(df_tratado.schema[c].dataType, DoubleType), c
assert isinstance(df_tratado.schema["data_inversa"].dataType, DateType)
for c in ["id", "pesid", "id_veiculo"]:
    assert isinstance(df_tratado.schema[c].dataType, StringType), c
for c in campos_veiculo:
    indicador = f"{c}_nao_se_aplica"
    assert isinstance(df_tratado.schema[indicador].dataType, BooleanType), indicador
    assert df_tratado.filter(F.col(indicador).isNull()).count() == 0, indicador
    assert df_tratado.filter(F.col(indicador) & F.col(c).isNotNull()).count() == 0, c

res_nulos = df_tratado.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"n{i}")
    for i, c in enumerate(colunas_originais)
]).first()
res_na = df_tratado.agg(*[
    F.sum(F.col(f"{c}_nao_se_aplica").cast("int")).alias(c)
    for c in campos_veiculo
]).first()
linhas = []
for i, c in enumerate(colunas_originais):
    nulos = int(res_nulos[f"n{i}"] or 0)
    nao_aplica = int(res_na[c] or 0) if c in campos_veiculo else 0
    assert nao_aplica <= nulos, c
    faltantes = nulos - nao_aplica
    aplicaveis = total_final - nao_aplica
    linhas.append((c, nulos, nao_aplica, faltantes, aplicaveis,
                   percentual(faltantes, aplicaveis) if aplicaveis else None))
df_resumo_ausencias_final = spark.createDataFrame(linhas,
    "coluna string, nulos_fisicos long, nao_se_aplica long, ausencias_aplicaveis long, registros_aplicaveis long, percentual_ausencia double")
print("Ausências FINAIS, inclusive as criadas por regras de invalidez/conversão:")
df_resumo_ausencias_final.orderBy(F.desc_nulls_last("percentual_ausencia")).show(len(linhas), False)
print("Registros:", total_final, "| Colunas:", len(df_tratado.columns))
print("Esquema após a tipagem:")
df_tratado.printSchema()
df_tratado.select("id", "pesid", "idade", "id_veiculo", "id_veiculo_nao_se_aplica",
                  "ano_fabricacao_veiculo", "ano_fabricacao_veiculo_nao_se_aplica").show(10, False)


Ausências FINAIS, inclusive as criadas por regras de invalidez/conversão:
+----------------------+-------------+-------------+--------------------+--------------------+-------------------+
|coluna                |nulos_fisicos|nao_se_aplica|ausencias_aplicaveis|registros_aplicaveis|percentual_ausencia|
+----------------------+-------------+-------------+--------------------+--------------------+-------------------+
|idade                 |106045       |0            |106045              |584010              |18.16              |
|sexo                  |93055        |0            |93055               |584010              |15.93              |
|estado_fisico         |91884        |0            |91884               |584010              |15.73              |
|pesid                 |56801        |0            |56801               |584010              |9.73               |
|tipo_envolvido        |56801        |0            |56801               |584010              |9.73               |
|ileso

**Resposta do grupo — limpeza e preparação.** A limpeza não exclui linhas nem preenche atributos pessoais por estimativa. Marcadores textuais de ausência tornam-se `NULL`; quatro indicadores preservam os casos de não aplicabilidade dos campos de veículo; valores incompatíveis de idade e ano de fabricação são invalidados com justificativa; os valores conhecidos são convertidos para tipos apropriados com auditoria. Foram preservadas idades iguais a zero e observações extremas plausíveis. Não se usou IQR para excluir dados: ele apenas orientou a inspeção, pois outlier estatístico não comprova erro. Também não se padronizou a ordem dos componentes de `tracado_via`, porque uma diferença de sequência não constitui, isoladamente, inconsistência comprovada.

Os percentuais finais são calculados **sobre atributos aplicáveis**, não sobre todos os `NULL` indistintamente. Essa escolha mantém a interpretação do desconhecimento e possibilita cálculos corretos posteriormente.

## 5. Transformação dos dados

Depois da limpeza, `df_tratado` já possui tipos adequados para idade, fabricação, coordenadas e datas, além dos indicadores que distinguem ausência de informação de não aplicabilidade. Não repetimos essas conversões: usamos seus resultados para derivar variáveis que serão necessárias às comparações temporais e às análises de acidentes e condutores.

As novas variáveis são criadas em `df_transformado`, mantendo `df_tratado` inalterado. `mes` e `hora` permitem segmentar ocorrências no tempo; os indicadores derivados tornam explícitas as regras de identificação de acidente fatal e condutor menor. Conjuntos auxiliares organizam registros pelas chaves apropriadas para que uma mesma ocorrência ou pessoa não seja contada repetidamente.

Um indicador só é atribuído quando os campos de origem sustentam a classificação. Caso a informação seja desconhecida, o resultado também permanece desconhecido, sem presumir que ocorreu ou não ocorreu o evento. Quando um indicador não é pertinente ao registro, a não aplicabilidade é representada de modo distinto.


### 5.1 — Enriquecimento temporal

**O que será criado:** `mes` a partir de `data_inversa` e `hora` a partir de `horario`, ambos inteiros. O dia da semana já existe no CSV e, portanto, **não será recriado**. Essas duas colunas dão suporte à segunda pergunta sem alterar a data e o horário originais. Se a data ou o horário não tiver valor válido, o derivado será `NULL`, em vez de inventar um período.

**Por que esta opção:** manter os campos originais e acrescentar duas variáveis simples facilita agrupamentos futuros e permite auditar a origem de cada resultado.

In [26]:
# 5.1 — Variáveis temporais, sem substituir data e horário existentes
df_transformado = (
    df_tratado
    .withColumn("mes", F.month(F.col("data_inversa")).cast("int"))
    .withColumn("hora", F.when(F.col("horario").isNotNull(),
                            F.substring(F.col("horario"), 1, 2).cast("int"))
                        .otherwise(F.lit(None).cast("int")))
)

assert df_transformado.count() == df_tratado.count(), "Perda de linhas no enriquecimento temporal"
assert df_transformado.filter(F.col("mes").isNotNull() & ~F.col("mes").between(1, 12)).count() == 0
assert df_transformado.filter(F.col("hora").isNotNull() & ~F.col("hora").between(0, 23)).count() == 0
print("Colunas derivadas: mes e hora. Dia da semana original preservado.")
df_transformado.select("id", "data_inversa", "mes", "dia_semana", "horario", "hora").show(8, False)

Colunas derivadas: mes e hora. Dia da semana original preservado.
+------+------------+---+------------+--------+----+
|id    |data_inversa|mes|dia_semana  |horario |hora|
+------+------------+---+------------+--------+----+
|652468|2025-01-01  |1  |quarta-feira|00:30:00|0   |
|652468|2025-01-01  |1  |quarta-feira|00:30:00|0   |
|652468|2025-01-01  |1  |quarta-feira|00:30:00|0   |
|652468|2025-01-01  |1  |quarta-feira|00:30:00|0   |
|652469|2025-01-01  |1  |quarta-feira|00:50:00|0   |
|652469|2025-01-01  |1  |quarta-feira|00:50:00|0   |
|652470|2025-01-01  |1  |quarta-feira|01:15:00|1   |
|652470|2025-01-01  |1  |quarta-feira|01:15:00|1   |
+------+------------+---+------------+--------+----+
only showing top 8 rows


### 5.2 — Indicadores derivados com semântica explícita

**Fatalidade:** `acidente_fatal` deriva da **classificação do acidente**, e não da soma das flags individuais `mortos`. Os valores serão `Sim`, `Não` ou `Não informado` conforme as categorias efetivamente existentes; não se infere uma classificação ausente a partir de um único indicador pessoal. A coluna serve como base para a terceira pergunta, respeitando que o acidente pode aparecer em mais de uma linha.

**Condutor menor de idade:** `condutor_menor` será `Sim` para condutor com idade válida inferior a 18, `Não` para condutor com 18 anos ou mais, `Não informado` quando falta idade ou tipo de envolvimento, e `Não se aplica` para pessoas comprovadamente de outra categoria. Essa distinção é importante na quinta pergunta: desconhecer a idade de um condutor não é o mesmo que afirmar que ele não é menor.

**Por que manter os indicadores como texto:** são categorias interpretáveis, não totais de acidentes nem números de vítimas. Os 0/1 originais de vítima continuam intactos, pois descrevem cada pessoa.

In [28]:
# 5.2 — Categorias derivadas sem sobrescrever dados tratados
classe = F.lower(F.trim(F.col("classificacao_acidente")))
papel = F.lower(F.trim(F.col("tipo_envolvido")))

df_transformado = (
    df_transformado
    .withColumn(
        "acidente_fatal",
        F.when(F.col("classificacao_acidente").isNull(), F.lit("Não informado"))
         .when(classe == "com vítimas fatais", F.lit("Sim"))
         .when(classe.isin("com vítimas feridas", "sem vítimas"), F.lit("Não"))
         .otherwise(F.lit("Não informado"))
    )
    .withColumn(
        "condutor_menor",
        F.when(F.col("tipo_envolvido").isNull(), F.lit("Não informado"))
         .when(papel != "condutor", F.lit("Não se aplica"))
         .when(F.col("idade").isNull(), F.lit("Não informado"))
         .when(F.col("idade") < 18, F.lit("Sim"))
         .otherwise(F.lit("Não"))
    )
)
print("Classes existentes no campo original de classificação:")
df_tratado.groupBy("classificacao_acidente").count().show(20, False)
print("Distribuição dos indicadores derivados por linha (não por acidente distinto):")
df_transformado.groupBy("acidente_fatal").count().show(truncate=False)
df_transformado.groupBy("condutor_menor").count().show(truncate=False)
print("Exemplos por registro:")
df_transformado.select("id", "classificacao_acidente", "acidente_fatal", "pesid", "tipo_envolvido", "idade", "condutor_menor").show(10, False)

Classes existentes no campo original de classificação:
+----------------------+------+
|classificacao_acidente|count |
+----------------------+------+
|NULL                  |7     |
|Com Vítimas Feridas   |435193|
|Com Vítimas Fatais    |94032 |
|Sem Vítimas           |54778 |
+----------------------+------+

Distribuição dos indicadores derivados por linha (não por acidente distinto):
+--------------+------+
|acidente_fatal|count |
+--------------+------+
|Não informado |7     |
|Não           |489971|
|Sim           |94032 |
+--------------+------+

+--------------+------+
|condutor_menor|count |
+--------------+------+
|Não informado |83723 |
|Não           |325744|
|Sim           |1336  |
|Não se aplica |173207|
+--------------+------+

Exemplos por registro:
+------+----------------------+--------------+-------+--------------+-----+--------------+
|id    |classificacao_acidente|acidente_fatal|pesid  |tipo_envolvido|idade|condutor_menor|
+------+----------------------+------------

### 5.3 — Preparação das unidades de observação

O CSV possui repetições legítimas de `id`, `pesid` e `tipo_acidente`. Um agrupamento indiscriminado por `id` descartaria os diferentes tipos de uma ocorrência; uma contagem de linhas poderia superestimar pessoas ou acidentes. Por isso, serão construídos **conjuntos auxiliares separados**, sem excluir nada do `df_transformado`:

- `df_acidentes`: uma linha por `id` conhecido, com características próprias da ocorrência. A unicidade das informações selecionadas é validada antes da redução; se existir conflito, o código interrompe a operação em vez de escolher um valor arbitrário.
- `df_acidente_tipo`: uma linha para cada par conhecido `id`–`tipo_acidente`. Preserva os vários tipos de um mesmo acidente, necessários à pergunta 3.
- `df_pessoas`: uma linha por `pesid` conhecido, com os atributos relevantes a pessoa e veículo. Registros sem `pesid` ficam no `df_sem_pesid`, evitando inventar identidades ou descartar linhas da origem.

**Por que essa separação:** cada pergunta usa uma unidade diferente. A transformação cria caminhos corretos para futuras contagens, sem antecipar respostas ou pressupor que repetição seja erro.

In [29]:
# 5.3 — Estruturas auxiliares por granularidade; nenhum registro é removido do df_transformado
atributos_acidente = ["data_inversa", "dia_semana", "horario", "uf", "br", "km",
                       "municipio", "classificacao_acidente", "fase_dia", "condicao_metereologica", "mes", "hora", "acidente_fatal"]

# countDistinct ignora NULL: por isso também contamos NULL como estado distinto.
verificacoes = [
    (F.countDistinct(F.col(c)) + F.max(F.col(c).isNull().cast("int"))).alias(f"estados_{i}")
    for i, c in enumerate(atributos_acidente)
]
ids_validos = df_transformado.filter(F.col("id").isNotNull())
conflitos = ids_validos.groupBy("id").agg(*verificacoes).filter(
    " OR ".join(f"estados_{i} > 1" for i in range(len(atributos_acidente)))
)
qtd_conflitos = conflitos.count()
print("IDs de acidente com divergência nos atributos estáveis selecionados:", qtd_conflitos)
assert qtd_conflitos == 0, "Investigar divergência antes de produzir df_acidentes"

df_acidentes = ids_validos.select("id", *atributos_acidente).dropDuplicates(["id"])
df_acidente_tipo = (ids_validos
    .filter(F.col("tipo_acidente").isNotNull())
    .select("id", "tipo_acidente").dropDuplicates())

# Conferimos os atributos de pessoa antes de reduzir sua repetição.
atributos_pessoa = ["id", "tipo_envolvido", "idade", "sexo", "estado_fisico",
                    "id_veiculo", "tipo_veiculo", "uf", "condutor_menor",
                    "ilesos", "feridos_leves", "feridos_graves", "mortos"]
com_pesid = df_transformado.filter(F.col("pesid").isNotNull())
exp_pessoa = [
    (F.countDistinct(F.col(c)) + F.max(F.col(c).isNull().cast("int"))).alias(f"estados_{i}")
    for i,c in enumerate(atributos_pessoa)
]
conflitos_pessoa = com_pesid.groupBy("pesid").agg(*exp_pessoa).filter(
    " OR ".join(f"estados_{i} > 1" for i in range(len(atributos_pessoa)))
)
qtd_conflitos_pessoa = conflitos_pessoa.count()
print("IDs de pessoa com divergência nos atributos selecionados:", qtd_conflitos_pessoa)
assert qtd_conflitos_pessoa == 0, "Investigar divergência antes de produzir df_pessoas"

df_pessoas = com_pesid.select("pesid", *atributos_pessoa).dropDuplicates(["pesid"])
df_sem_pesid = df_transformado.filter(F.col("pesid").isNull())
print("Acidentes distintos:", df_acidentes.count())
print("Pares acidente-tipo:", df_acidente_tipo.count())
print("Pessoas identificadas distintas:", df_pessoas.count())
print("Linhas sem pesid mantidas no conjunto auxiliar:", df_sem_pesid.count())
print("Todos os registros continuam disponíveis em df_transformado:", df_transformado.count())

IDs de acidente com divergência nos atributos estáveis selecionados: 0
IDs de pessoa com divergência nos atributos selecionados: 0
Acidentes distintos: 72529
Pares acidente-tipo: 122569
Pessoas identificadas distintas: 177479
Linhas sem pesid mantidas no conjunto auxiliar: 56801
Todos os registros continuam disponíveis em df_transformado: 584010


### 5.4 — Conferência das transformações

A verificação final testa se o enriquecimento temporal e os indicadores **não alteraram a quantidade de linhas**, se as novas colunas existem com valores esperados e se cada tabela auxiliar respeita a chave escolhida. Ela também compara a presença de identificadores para confirmar que os registros sem `pesid` não desapareceram: permanecerão em `df_transformado` e no conjunto auxiliar específico.

Esses testes são necessários porque uma transformação útil somente é confiável quando se pode demonstrar que não introduziu duplicação indevida, perda silenciosa ou categorias contraditórias.

In [30]:
# 5.4 — Validação dos produtos de transformação
assert df_transformado.count() == df_tratado.count(), "Linhas do conjunto transformado divergentes"
assert df_transformado.columns[:len(df_tratado.columns)] == df_tratado.columns
assert len(df_transformado.columns) == len(df_tratado.columns) + 4
assert df_acidentes.groupBy("id").count().filter("count > 1").count() == 0
assert df_acidente_tipo.groupBy("id", "tipo_acidente").count().filter("count > 1").count() == 0
assert df_pessoas.groupBy("pesid").count().filter("count > 1").count() == 0
assert df_pessoas.count() == com_pesid.select("pesid").distinct().count()
assert df_sem_pesid.count() == df_transformado.filter(F.col("pesid").isNull()).count()
for c, opcoes in {"acidente_fatal": ["Sim", "Não", "Não informado"],
                  "condutor_menor": ["Sim", "Não", "Não informado", "Não se aplica"]}.items():
    assert df_transformado.filter(F.col(c).isNull() | ~F.col(c).isin(opcoes)).count() == 0, c
print("Validações aprovadas. Produtos preparados: df_tratado, df_transformado, df_acidentes, df_acidente_tipo, df_pessoas, df_sem_pesid.")
print("Exemplo de colunas derivadas:")
df_transformado.select("id", "mes", "hora", "acidente_fatal", "condutor_menor").show(8, False)

Validações aprovadas. Produtos preparados: df_tratado, df_transformado, df_acidentes, df_acidente_tipo, df_pessoas, df_sem_pesid.
Exemplo de colunas derivadas:
+------+---+----+--------------+--------------+
|id    |mes|hora|acidente_fatal|condutor_menor|
+------+---+----+--------------+--------------+
|652468|1  |0   |Não           |Não informado |
|652468|1  |0   |Não           |Não se aplica |
|652468|1  |0   |Não           |Não           |
|652468|1  |0   |Não           |Não se aplica |
|652469|1  |0   |Não           |Não se aplica |
|652469|1  |0   |Não           |Não           |
|652470|1  |1   |Não           |Não se aplica |
|652470|1  |1   |Não           |Não se aplica |
+------+---+----+--------------+--------------+
only showing top 8 rows


**Resposta do grupo — transformações realizadas.** A partir da base limpa, foram criadas variáveis mensais e horárias sem duplicar o dia da semana já fornecido. Os indicadores de fatalidade e de condutor menor foram definidos com estados explícitos para informações desconhecidas e casos não aplicáveis; os indicadores pessoais 0/1 foram mantidos com seu significado original. Como a extração contém múltiplas linhas por ocorrência, foram produzidas representações auxiliares por acidente, acidente–tipo e pessoa, com testes de consistência antes de qualquer redução de linhas. O conjunto completo permanece em `df_transformado`, preservando informações da origem. As transformações facilitam responder às perguntas futuramente, mas seus resultados aqui têm caráter de preparação e validação, não de conclusão analítica.

## 6. Conjuntos de dados resultantes

**`df_tratado`:** contém as 37 colunas originais com os tipos definidos na limpeza e quatro indicadores booleanos de não aplicabilidade veicular. **`df_transformado`:** acrescenta `mes`, `hora`, `acidente_fatal` e `condutor_menor`, sem remover os registros. **Conjuntos auxiliares:** `df_acidentes`, `df_acidente_tipo`, `df_pessoas` e `df_sem_pesid` organizam as observações segundo seus identificadores e preservam os casos sem identificação de pessoa para análises sobre o acidente.

A célula seguinte apresenta os DataFrames e seus esquemas, para explicitar o conteúdo disponível após o processamento. A gravação em Parquet pode ser ativada para reutilizar os dados com seus tipos nativos e valores `NULL`; quando desativada, nenhum arquivo é criado. Essa distinção evita afirmar que houve exportação sem que a operação tenha sido executada.


In [33]:
# 6 — Identificação dos dados resultantes e materialização opcional
print("Dataset limpo: df_tratado | linhas:", df_tratado.count(), "| colunas:", len(df_tratado.columns))
print("Dataset transformado: df_transformado | linhas:", df_transformado.count(), "| colunas:", len(df_transformado.columns))
print("Tipos finais do conjunto transformado:")
df_transformado.printSchema()

EXPORTAR_PARQUET =  True
CAMINHO_PARQUET = "/content/ra1_acidentes_2025_transformados_parquet"
if EXPORTAR_PARQUET:
    df_transformado.write.mode("overwrite").parquet(CAMINHO_PARQUET)
    df_recarregado = spark.read.parquet(CAMINHO_PARQUET)
    assert df_recarregado.count() == df_transformado.count(), "Quantidade de linhas divergente"

    # Compara apenas o nome e o tipo de dados de cada coluna, ignorando a propriedade 'nullable'
    schema_recarregado = [(field.name, field.dataType) for field in df_recarregado.schema]
    schema_transformado = [(field.name, field.dataType) for field in df_transformado.schema]
    assert schema_recarregado == schema_transformado, "Esquema divergente em nomes ou tipos de dados"

    print("Arquivo Parquet produzido e validado:", CAMINHO_PARQUET)
else:
    print("Exportação não acionada: conjuntos disponíveis como DataFrames nesta sessão Spark.")

Dataset limpo: df_tratado | linhas: 584010 | colunas: 41
Dataset transformado: df_transformado | linhas: 584010 | colunas: 45
Tipos finais do conjunto transformado:
root
 |-- id: string (nullable = true)
 |-- pesid: string (nullable = true)
 |-- data_inversa: date (nullable = true)
 |-- dia_semana: string (nullable = true)
 |-- horario: string (nullable = true)
 |-- uf: string (nullable = true)
 |-- br: integer (nullable = true)
 |-- km: double (nullable = true)
 |-- municipio: string (nullable = true)
 |-- causa_principal: string (nullable = true)
 |-- causa_acidente: string (nullable = true)
 |-- ordem_tipo_acidente: integer (nullable = true)
 |-- tipo_acidente: string (nullable = true)
 |-- classificacao_acidente: string (nullable = true)
 |-- fase_dia: string (nullable = true)
 |-- sentido_via: string (nullable = true)
 |-- condicao_metereologica: string (nullable = true)
 |-- tipo_pista: string (nullable = true)
 |-- tracado_via: string (nullable = true)
 |-- uso_solo: string (nul

## 7. Síntese dos resultados e decisões

**Estrutura observada.** O conjunto de 2025 contém 584.010 linhas e 37 colunas originais. Há acidentes associados a várias pessoas, causas e tipos; por isso, repetições de `id` não foram consideradas duplicações indevidas. Para preservar a informação de diferentes tipos registrados no mesmo acidente, foi mantida a relação acidente–tipo. Os campos `mortos`, `ilesos`, `feridos_leves` e `feridos_graves` representam situações individuais (0/1), de modo que pessoas do mesmo acidente podem apresentar valores diferentes sem contradição.

**Ausências e plausibilidade.** As diferentes grafias usadas para indicar desconhecimento foram normalizadas para `NULL`. Quatro indicadores booleanos permitem distinguir campos veiculares não aplicáveis de informações aplicáveis que não foram fornecidas; essa distinção também orienta o cálculo dos percentuais de ausência. Foram considerados inválidos valores negativos ou superiores a 110 para idade e anos de fabricação não positivos, conforme as verificações exploratórias. A idade zero foi mantida por poder representar crianças com menos de um ano. Não foram inventadas idades, identificadores ou classificações para preencher campos desconhecidos.

**Valores extremos.** O IQR foi utilizado para localizar observações merecedoras de inspeção, não para eliminar automaticamente registros. Um valor raro pode ser legítimo e sua exclusão indiscriminada distorceria a distribuição dos acidentes. O campo `tracado_via` foi preservado na forma registrada pela fonte, pois não há comprovação de que a ordem dos seus componentes seja um erro. Registros sem classificação do acidente foram mantidos com essa limitação indicada, sem inferir sua classe apenas de indicadores individuais.

**Resultado do processamento.** `df_tratado` reúne os atributos limpos e tipados; `df_transformado` acrescenta variáveis temporais e indicadores definidos a partir dos campos disponíveis. Os conjuntos auxiliares organizam os registros por acidente, por acidente–tipo e por pessoa, sem descartar os registros sem `pesid` do conjunto completo. Os testes de integridade e as tabelas de saída permitem verificar perdas indevidas, conflitos de chave e o efeito das regras aplicadas. A interpretação de frequências e proporções deve considerar a unidade de observação e a cobertura de cada atributo.


# RA2 — Streaming Data Processing and Time Windows

Utilize os dados e o contexto do projeto para demonstrar o processamento em streaming e a aplicação de janelas temporais. Escolha um dos cenários:

- **Cenário A — Dados estáticos:** simule a chegada de novos registros utilizando dados artificiais coerentes com o contexto do projeto.


## 8. Objetivo da análise em streaming

Responda brevemente:

- O que o grupo deseja monitorar ou analisar?
- Qual informação espera obter com o processamento em streaming?

**Resposta do grupo:**

O objetivo é monitorar a evolução diária dos acidentes simulados em rodovias federais à medida que novos registros são recebidos. Para cada dia de ocorrência, queremos identificar (1) o número de acidentes, (2) a quantidade de acidentes classificados como fatais, (3) quantos registros envolvem condutores menores de 18 anos e (4) o percentual de acidentes fatais.

A atualização gradual permite observar como os indicadores mudam após cada chegada e verificar se um registro enviado com atraso é contabilizado no dia em que o acidente ocorreu, e não no dia da recepção. Os registros de 2026 são inteiramente fictícios e servem apenas para demonstrar o processamento incremental.

## 9. Fonte de dados e processamento

- Informe o cenário escolhido: dados reais em streaming ou simulação.
- Explique brevemente como os novos registros chegam.
- Implemente o processamento em tempo real.
- Demonstre o recebimento de novos registros.

**Resposta do grupo:**

Cenário escolhido: A — simulação com dados estáticos. O RA1 utiliza o conjunto histórico de acidentes de 2025 da Polícia Rodoviária Federal (PRF); não dispomos de uma fonte contínua de novos registros. Por isso, simulamos 12 acidentes distintos de outubro de 2026 em quatro lotes sucessivos, com três registros em cada lote.

Para tornar os exemplos coerentes com o contexto do projeto, os atributos gerais foram copiados de registros da base de 2025 e adaptados com IDs fictícios, datas e horários de 2026, identificação de condutores e classificações escolhidas para a demonstração. Cada registro preserva as 37 colunas do CSV original, às quais é acrescentado instante_chegada, o horário de recepção simulado. O campo momento_acidente é calculado a partir de data_inversa e horario e determina o agrupamento temporal; ele não é o mesmo que instante_chegada.

Cada lote é gravado como um arquivo JSON Lines na pasta monitorada pelo spark.readStream, com esquema definido previamente e maxFilesPerTrigger=1. O arquivo é preparado fora da pasta de entrada e movido para ela somente quando está completo. O Spark utiliza writeStream e checkpoint para atualizar as agregações sem precisar executar novamente uma leitura batch de toda a pasta a cada chegada.

O terceiro lote contém um registro de 19/10/2026 que chega depois de registros de 20/10/2026. Isso permitirá mostrar que uma janela previamente exibida pode ser atualizada pela chegada tardia de um evento.

Tratamento de dados: diferentemente do RA1, o foco desta etapa não é limpar a base histórica. Aplicamos somente as conversões indispensáveis às agregações: criação do timestamp do acidente, classificação de fatalidade e identificação de condutor menor. As demais 37 colunas são mantidas na entrada do fluxo. O conjunto fictício contém uma linha por acidente; essa escolha não deve ser generalizada para a base histórica, onde o mesmo id pode aparecer em várias linhas.

In [46]:
# 9.1 — Preparação da simulação (execute após a seção do RA1 que cria df_original)
from pathlib import Path
import json
import shutil
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, TimestampType

RA2_DIR = Path('/content/ra2_streaming_acidentes')
ENTRADA = RA2_DIR / 'entrada'
CHECKPOINT = RA2_DIR / 'checkpoint'

# Reinicie apenas as consultas do RA2 antes de limpar os arquivos da simulação
for q in list(spark.streams.active):
    if q.name == 'ra2_janelas_acidentes':
        q.stop()
shutil.rmtree(RA2_DIR, ignore_errors=True)
ENTRADA.mkdir(parents=True, exist_ok=True)
spark.conf.set('spark.sql.shuffle.partitions', '4')

colunas_originais = df_original.columns
assert len(colunas_originais) == 37, 'Verifique se df_original é a base original de 37 colunas.'
esquema_ra2 = StructType(
    list(df_original.schema.fields) + [StructField('instante_chegada', TimestampType(), True)]
)
print('Colunas originais:', len(colunas_originais))
print('Colunas de entrada no streaming:', len(esquema_ra2.fields))


Colunas originais: 37
Colunas de entrada no streaming: 38


**Coerência temporal da simulação:** Para garantir a consistência cronológica, os quatro lotes foram simulados como recebidos em 22/10/2026, com horários crescentes entre 14h e 14h11. Dessa forma, todos os registros são recebidos após a ocorrência dos respectivos acidentes. Além disso, um acidente ocorrido em 19/10/2026 foi intencionalmente incluído no terceiro lote, permitindo demonstrar como o Spark Structured Streaming atualiza uma janela temporal anterior quando recebe um registro atrasado.

In [47]:

# 9.2 — Criação de 12 registros fictícios para o streaming

from datetime import datetime, timedelta

# Seleciona 12 acidentes distintos como modelos
modelos = []
ids_vistos = set()

for linha in df_original.limit(300).collect():
    dados = linha.asDict(recursive=True)

    if dados.get('id') not in ids_vistos:
        ids_vistos.add(dados.get('id'))
        modelos.append(dados)

    if len(modelos) == 12:
        break

assert len(modelos) == 12, (
    'Não foram encontrados 12 acidentes diferentes na amostra inicial.'
)

# Distribuição dos acidentes por dia:
# 19/10: 4 | 20/10: 5 | 21/10: 3
dias_evento = [19, 19, 19, 20, 20, 20, 20, 20, 21, 19, 21, 21]

# Ordem de recepção, respeitando os quatro lotes da seção 11
ordem_chegada = [0, 1, 2, 3, 4, 5, 6, 7, 9, 8, 10, 11]

# Simulação de recebimento em 22/10, após todos os acidentes
inicio_chegada = datetime(2026, 10, 22, 14, 0, 0)

horarios_chegada = {
    indice: (
        inicio_chegada + timedelta(minutes=posicao)
    ).strftime('%Y-%m-%d %H:%M:%S')
    for posicao, indice in enumerate(ordem_chegada)
}

registros_ra2 = []

for i, (modelo, dia) in enumerate(zip(modelos, dias_evento), 1):
    r = dict(modelo)

    # Identificadores artificiais
    r['id'] = f'SIM2026-{i:03d}'
    r['pesid'] = f'PES-SIM-{i:03d}'
    r['id_veiculo'] = f'VEI-SIM-{i:03d}'

    # Data e horário fictícios da ocorrência
    r['data_inversa'] = f'2026-10-{dia:02d}'
    r['dia_semana'] = {
        19: 'segunda-feira',
        20: 'terça-feira',
        21: 'quarta-feira'
    }[dia]

    r['horario'] = (
        f'{8 + (i % 10):02d}:{(i * 7) % 60:02d}:00'
    )

    # Indicadores controlados para a análise
    fatal = i in (5, 11)
    menor = i == 7

    r['mortos'] = '1' if fatal else '0'
    r['classificacao_acidente'] = (
        'Com Vítimas Fatais' if fatal else 'Sem Vítimas'
    )
    r['idade'] = '17' if menor else str(25 + i)
    r['tipo_envolvido'] = 'Condutor'
    r['estado_fisico'] = 'Óbito' if fatal else 'Ileso'
    r['ilesos'] = '0' if fatal else '1'
    r['feridos_leves'] = '0'
    r['feridos_graves'] = '0'

    # Horário de chegada respeitando a ordem dos lotes
    r['instante_chegada'] = horarios_chegada[i - 1]

    registros_ra2.append(r)

# Validações da simulação
assert len({r['id'] for r in registros_ra2}) == 12

assert all(
    set(r) == set(colunas_originais) | {'instante_chegada'}
    for r in registros_ra2
)

# Nenhum registro pode chegar antes de acontecer
assert all(
    datetime.strptime(
        r['instante_chegada'], '%Y-%m-%d %H:%M:%S'
    ) >= datetime.strptime(
        r['data_inversa'] + ' ' + r['horario'],
        '%Y-%m-%d %H:%M:%S'
    )
    for r in registros_ra2
)

# Confere se os horários seguem a sequência de recepção
chegadas_ordenadas = [
    registros_ra2[indice]['instante_chegada']
    for indice in ordem_chegada
]

assert chegadas_ordenadas == sorted(chegadas_ordenadas)

print('Acidentes simulados:', len(registros_ra2))
print('Datas das ocorrências: 19, 20 e 21/10/2026')
print('Data de recebimento: 22/10/2026')
print('Validação cronológica concluída com sucesso!')


Acidentes simulados: 12
Datas das ocorrências: 19, 20 e 21/10/2026
Data de recebimento: 22/10/2026
Validação cronológica concluída com sucesso!


In [48]:
# 9.3 — Fonte incremental: arquivo JSON recebido => micro-lote Spark
fluxo_bruto = (
    spark.readStream
         .schema(esquema_ra2)
         .option('maxFilesPerTrigger', 1)
         .json(str(ENTRADA))
)

# Somente as conversões indispensáveis, sem refazer o tratamento do RA1
fluxo_ra2 = (
    fluxo_bruto
    .withColumn('momento_acidente',
        F.to_timestamp(F.concat_ws(' ', F.col('data_inversa'), F.col('horario')),
                       'yyyy-MM-dd HH:mm:ss'))
    .withColumn('acidente_fatal_ra2',
        F.when(F.lower(F.trim(F.col('classificacao_acidente'))) == 'com vítimas fatais', 1).otherwise(0))
    .withColumn('condutor_menor_ra2',
        F.when(
            (F.lower(F.trim(F.col('tipo_envolvido'))) == 'condutor') &
            (F.expr('try_cast(trim(idade) as int)') < 18) &
            (F.expr('try_cast(trim(idade) as int)') >= 0), 1
        ).otherwise(0))
)
print('É streaming?', fluxo_ra2.isStreaming)
print('Campos na leitura:', len(fluxo_bruto.columns))


É streaming? True
Campos na leitura: 38


## 10. Aplicação de janelas temporais

- Utilize uma janela **Tumbling** (intervalos sem sobreposição) ou **Sliding** (intervalos que podem se sobrepor).
- Informe o tipo e a duração da janela e o deslocamento, quando aplicável.
- Implemente uma agregação coerente com a análise, como contagem, soma ou média.
- Explique brevemente o que será calculado em cada intervalo.

O foco é demonstrar a compreensão do conceito; não é necessário justificar tecnicamente a duração escolhida.

**Resposta do grupo:**

Escolhemos uma janela Tumbling de 1 dia, aplicada a momento_acidente (event time), que representa a data e a hora da ocorrência. As janelas não se sobrepõem; seu deslocamento é de 1 dia, igual à duração. Assim, cada acidente simulado pertence a exatamente uma janela diária.

Em cada janela calculamos:
- acidentes — contagem dos registros de acidentes do dia; como os 12 IDs simulados são únicos e cada acidente tem apenas uma linha, essa contagem equivale à quantidade de acidentes distintos nesta simulação;
- acidentes_fatais — quantidade classificada como Com Vítimas Fatais;
- condutores_menores — quantidade de registros de condutores com idade válida inferior a 18 anos;
- pct_fatais — proporção percentual de acidentes classificados como fatais entre os registros do dia.
Usamos outputMode('complete') e uma tabela em memória para facilitar a apresentação: após cada micro-lote, podemos consultar os resultados de todas as janelas. O registro atrasado será acrescentado à janela da data do acidente. Esse modo é adequado à demonstração pequena e finita, não a uma execução ilimitada em produção.

In [49]:
# 10 — Agregações em janelas Tumbling de 1 dia
janelas_ra2 = (
    fluxo_ra2
    .groupBy(F.window('momento_acidente', '1 day').alias('janela'))
    .agg(
        F.count('*').alias('acidentes'),
        F.sum('acidente_fatal_ra2').alias('acidentes_fatais'),
        F.sum('condutor_menor_ra2').alias('condutores_menores')
    )
    .withColumn('pct_fatais',
        F.round(100 * F.col('acidentes_fatais') / F.col('acidentes'), 1))
)

consulta_ra2 = (
    janelas_ra2.writeStream
      .format('memory')
      .queryName('ra2_janelas_acidentes')
      .outputMode('complete')
      .option('checkpointLocation', str(CHECKPOINT))
      .start()
)
print('Consulta de janelas ativa:', consulta_ra2.isActive)


Consulta de janelas ativa: True


## 11. Resultados e interpretação

Apresente os resultados do processamento e demonstre como a análise se comporta com a chegada de novos registros.

**Resposta do grupo:**

A consulta de streaming foi executada com quatro lotes de três registros cada. As saídas registradas mostram evolução coerente com a chegada gradual dos dados:

| Etapa | 19/10/2026 | 20/10/2026 | 21/10/2026 | Total acumulado |
|---|---:|---:|---:|---:|
| Lote 1 | 3 | — | — | 3 |
| Lote 2 | 3 | 3 | — | 6 |
| Lote 3 | 4 | 5 | — | 9 |
| Lote 4 | 4 | 5 | 3 | 12 |


O lote 3 evidencia o principal comportamento temporal: a contagem de 19/10 passa de 3 para 4, pois chegou tardiamente um acidente dessa data; ao mesmo tempo, a contagem de 20/10 passa de 3 para 5. O Spark não atribuiu o evento atrasado ao dia da chegada: utilizou momento_acidente para posicioná-lo na janela correta.

Ao final, 19/10 apresenta 4 acidentes e nenhum fatal; 20/10, 5 acidentes, 1 fatal e 1 registro de condutor menor (20,0% de fatais); 21/10, 3 acidentes e 1 fatal (33,3% de fatais). No conjunto, foram contabilizados 12 acidentes fictícios e 2 fatais. Esses valores são resultados didáticos da simulação, não estatísticas oficiais da PRF.

Para verificar a consistência, os mesmos JSONs foram lidos em modo batch e agregados com as mesmas regras. A execução registrada apresentou 12 registros no streaming e nenhuma diferença entre as tabelas de streaming e batch. Essa comparação indica que, nesta demonstração, os lotes foram contabilizados corretamente, sem perda ou duplicação no resultado final.

In [50]:
# 11.1 — Chegada gradual dos quatro lotes e resultados intermediários
# Índices Python dos registros: o registro 9 (índice 9) é o evento atrasado de 19/10.
lotes = [
    [0, 1, 2],       # 19/10
    [3, 4, 5],       # 20/10
    [6, 7, 9],       # 20/10 + 1 evento atrasado de 19/10
    [8, 10, 11]      # 21/10
]

def mostrar_janelas():
    spark.sql("""SELECT date_format(janela.start, 'dd/MM/yyyy') AS dia,
                         acidentes, acidentes_fatais, condutores_menores, pct_fatais
                  FROM ra2_janelas_acidentes ORDER BY janela.start""").show(truncate=False)

for numero, indices in enumerate(lotes, start=1):
    dados_lote = [registros_ra2[i] for i in indices]
    temporario = RA2_DIR / f'lote_{numero}.tmp'
    # JSON Lines: um registro por linha, todas as 37 colunas + chegada
    with temporario.open('w', encoding='utf-8') as f:
        for registro in dados_lote:
            f.write(json.dumps(registro, ensure_ascii=False, default=str) + '\n')
    temporario.replace(ENTRADA / f'lote_{numero}.json')

    consulta_ra2.processAllAvailable()
    progresso = consulta_ra2.lastProgress
    qtd_processada = progresso['numInputRows'] if progresso else None
    print(f'\nLOTE {numero} — enviado: {len(dados_lote)}; processado: {qtd_processada}')
    mostrar_janelas()



LOTE 1 — enviado: 3; processado: 3
+----------+---------+----------------+------------------+----------+
|dia       |acidentes|acidentes_fatais|condutores_menores|pct_fatais|
+----------+---------+----------------+------------------+----------+
|19/10/2026|3        |0               |0                 |0.0       |
+----------+---------+----------------+------------------+----------+


LOTE 2 — enviado: 3; processado: 3
+----------+---------+----------------+------------------+----------+
|dia       |acidentes|acidentes_fatais|condutores_menores|pct_fatais|
+----------+---------+----------------+------------------+----------+
|19/10/2026|3        |0               |0                 |0.0       |
|20/10/2026|3        |1               |0                 |33.3      |
+----------+---------+----------------+------------------+----------+


LOTE 3 — enviado: 3; processado: 3
+----------+---------+----------------+------------------+----------+
|dia       |acidentes|acidentes_fatais|condutores_

In [51]:
# 11.2 — Validação: streaming e processamento em lote devem coincidir
resultado_streaming = spark.table('ra2_janelas_acidentes')

# Ler todos os JSONs da simulação de uma só vez usando Spark em modo batch
em_lote = (
    spark.read.schema(esquema_ra2).json(str(ENTRADA))
    .withColumn('momento_acidente',
        F.to_timestamp(F.concat_ws(' ', F.col('data_inversa'), F.col('horario')),
                       'yyyy-MM-dd HH:mm:ss'))
    .withColumn('acidente_fatal_ra2',
        F.when(F.lower(F.trim(F.col('classificacao_acidente'))) == 'com vítimas fatais', 1).otherwise(0))
    .withColumn('condutor_menor_ra2',
        F.when((F.lower(F.trim('tipo_envolvido')) == 'condutor') &
               (F.expr('try_cast(trim(idade) as int)') < 18) &
               (F.expr('try_cast(trim(idade) as int)') >= 0), 1).otherwise(0))
    .groupBy(F.window('momento_acidente', '1 day').alias('janela'))
    .agg(F.count('*').alias('acidentes'),
         F.sum('acidente_fatal_ra2').alias('acidentes_fatais'),
         F.sum('condutor_menor_ra2').alias('condutores_menores'))
    .withColumn('pct_fatais',
        F.round(100 * F.col('acidentes_fatais') / F.col('acidentes'), 1))
)

diferencas = (
    resultado_streaming.exceptAll(em_lote).count() +
    em_lote.exceptAll(resultado_streaming).count()
)
total = resultado_streaming.agg(F.sum('acidentes')).first()[0]
print('Acidentes simulados:', len(registros_ra2))
print('Acidentes contados no streaming:', total)
print('Linhas diferentes entre streaming e batch:', diferencas)
assert total == 12 and diferencas == 0
print('Validação concluída.')
consulta_ra2.stop()


Acidentes simulados: 12
Acidentes contados no streaming: 12
Linhas diferentes entre streaming e batch: 0
Validação concluída.


## 12. Considerações do RA2

Responda brevemente:

- O que foi possível observar com o processamento em streaming?
- Qual foi a contribuição das janelas temporais para a análise?
- Essa abordagem faria sentido em uma aplicação real do projeto? Por quê?

Na última resposta, o grupo pode reconhecer que o streaming foi utilizado apenas como exercício didático, se esse for o caso.

**Resposta do grupo:**

O que observamos: o Structured Streaming acompanhou a pasta de entrada, recebeu os quatro arquivos sucessivos e atualizou a tabela de indicadores após cada micro-lote. Foi possível comprovar tanto o crescimento das contagens quanto a atualização de uma janela anterior após a chegada atrasada de um evento. A comparação com o processamento batch confirmou a consistência do resultado final da simulação.

Contribuição das janelas temporais: a janela Tumbling de um dia transformou registros individuais em indicadores diários comparáveis. Ao utilizar o momento de ocorrência, e não o horário de ingestão, o Spark atribuiu cada evento à data adequada, inclusive no caso de atraso. A contagem de acidentes fatais e o percentual diário oferecem uma visão simples da gravidade das ocorrências recebidas.

Aplicação prática e limitações: esse processamento poderia ser útil em um sistema de monitoramento de acidentes alimentado continuamente, especialmente para acompanhar mudanças rápidas na frequência ou gravidade dos registros. Entretanto, neste RA2 a fonte é estática e os eventos de 2026 foram simulados para fins acadêmicos. A demonstração é pequena, utiliza tabela em memória e modo complete; uma aplicação real demandaria armazenamento persistente, regras de qualidade, controle de duplicidades, tratamento de atrasos e configuração de retenção de estado. Os resultados simulados não devem ser utilizados para inferir tendências reais de acidentes em 2026.